# Orbit Wars -- v5 TARGET actor + redesigned reward (pure-PyTorch PPO)

A **single, self-contained** notebook that trains the Orbit Wars **v5 target actor** of
`docs/set-ups/1.md` in **pure PyTorch** -- no native C++/LibTorch build, no
`kaggle_environments`. It runs on a simple cloud GPU (Colab / Kaggle Notebooks). It is a faithful
port of the native trainer (`gpu_env.cpp`, `policy_net.cpp`, `distribution.hpp`, `rollout.cpp`,
`grpo_trainer.cpp`) and matches `scripts/run_setup1_target.cmd`.

## How to run
1. (Colab/Kaggle) run the **pip** cell if torch is missing, then **Run all**.
2. The **config** cell has a single `SMOKE` boolean at the top: `SMOKE=True` is a fast
   end-to-end sanity run (tiny model/batch, 3 iters reaching stage-4 self-play); `SMOKE=False`
   (the delivered default) is the full `docs/set-ups/1.md` spec (width 512, 6 res blocks, B=128,
   curriculum 0/100/300/1400). Checkpoints are written to `/kaggle/working` or `/content` (or `.`).

## The v5 TARGET actor (the key change vs the continuous actor)
Per planet, **one launch per step**. The action per planet is **(destination, phi)**:
- **destination**: a **categorical over the E planet slots** (the obs rows), MASKED to alive
  planets (dead-destination logits -> `-1e9` before softmax). Head: `Linear(d+d_g -> E)`.
- **phi**: a continuous launch fraction in (0,1) = % of THAT planet's ships -- a **squashed
  Gaussian** `phi = sigmoid(N(mu, sigma^2))`, exactly like the old phi. `mu` head is
  `Linear(d+d_g -> 1)` (init bias `-4`, weights `*0.02`); the phi log-std is a state-dependent
  head clipped to `[-2, logstd_max]`.
- The trunk (per-planet proj + cross-planet self-attention + GLU + residual blocks + globals
  embedding) is **UNCHANGED**; only the heads change.

**Decode** (ego AND the self-play opponent): for each owned planet with ships>0, commit iff
`phi >= act_threshold (0.05)`; `n = floor(phi * ships)`; `dest = round(action[...,0])` clamped to
`[0,E-1]`; **heading = atan2(p_dest - p_src)** (aim straight at the destination). The destination
categorical is sized to `E = PLANET_CAP`, so the head's output dim equals the env's planet-slot count.

## NEW reward (replaces the old event reward entirely, `docs/set-ups/1.md`)
1. **win** `+1000`, **loss** `-1000`, draw `0`.
2. win/loss are **FLAT for the first 100 steps**, then decay symmetrically:
   `value = (+/-)1000 * 0.995^max(0, len - 100)` (`DECAY_START_STEP=100`, `WIN_DECAY=LOSS_DECAY=0.995`).
3. **capture a planet `+30`, lose a planet `-30`** (per ego owner-flip among alive planets, per step).
4. **first 50 VALID launches `+1`** each. A VALID launch (notebook simplification) = a committed
   launch (`phi>=thr`, `n>=1`) whose chosen destination is an **alive** planet. Cap at 50/game.
5. **production milestone `+20`** every time the ego's TOTAL ship count (owned planets + ego fleets
   in flight) crosses a doubling threshold from `base=100`: 100, 200, 400, ... Monotonic
   (`m_now = floor(log2(s/base))+1` for `s>=base`; reward `20 * relu(m_now - m_prev)`), so a ship
   loss never claws reward back.

The PPO per-step reward **and** the outcome are rescaled by `/100` before GAE (value-target
stability); advantages are re-normalized so the policy objective is unchanged. The per-iter
heartbeat logs the per-channel means `R[o c d p]` = outcome / capture / dispatch / prod-milestone
in **real (unscaled)** units.

## Curriculum (iteration-driven, 4 stages; matches `run_setup1_target.cmd`)
| stage | opponent | iters |
|-------|----------|-------|
| 1 | stationary / noop | 0 (skipped) |
| 2 | random | 1..100 |
| 3 | starter (nearest static non-owned) | 101..400 |
| 4 | self-play + starter + random (50/25/25), `selfplay_prob 0.5` | 401..1400 |

The self-play opponent is a **frozen snapshot** of the current policy, acting on the player-1
observation (`encode(ego=1)`), refreshed every 100 iters.

## Simplifications vs. the native engine
- **Comets are OMITTED** (`COMET_SLOTS = 0`; no comet schedule). Everything else (orbit, swept
  fleet/planet collision, OOB/sun removal, two-player combat) is ported 1:1.
- **VALID launch** uses the prompt's destination-is-alive definition (committed launch whose chosen
  destination slot is an alive planet), not the C++ ray-vs-disk landing test. This is the carrot
  for the first-50-dispatch channel only.
- **World generator** is a pure-Python/torch mirror of the official `generate_planets` (4-fold
  symmetric groups, `MIN/MAX_PLANET_GROUPS`, polar static groups, ships/production ranges) + home
  assignment; comet schedule dropped.

> Do NOT run heavy training on import. The shape smoke cell is guarded by `RUN_SMOKE=False`.

## 1. Imports
Only `torch`, `numpy`, `matplotlib` -- all preinstalled on Colab / Kaggle.

In [ ]:
# Colab/Kaggle usually ship torch already. Uncomment to (re)install a CUDA build if needed.
# import sys, subprocess
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'matplotlib'])
try:
    import torch  # noqa: F401
    print('torch present')
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'matplotlib'])

In [ ]:
import math
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32  # integer-in-float32 convention (mirrors the native env)
print("device:", DEVICE, "| torch", torch.__version__)

## 2. Config (editable hyperparameters)

Mirrors scripts/run_setup1_target.cmd (v5 target actor). Set SMOKE = True for a fast
Colab/Kaggle sanity run (tiny model, fewer iters, smaller batch); SMOKE = False (default) is
the full spec.

Full run uses B=128 (num_groups 16 * group_size 8), width 512, 6 res blocks. You can drop
B/HIDDEN to fit VRAM. PPO has no groups (the critic is the baseline), so B is just the env count.

In [ ]:
# ============================================================================
# CONFIG  --  set SMOKE=False for the real run (the delivered default).
# ============================================================================
# A SINGLE boolean controls everything: SMOKE=True shrinks the model/batch/iters
# for a fast end-to-end sanity run; SMOKE=False is the full docs/set-ups/1.md v5
# target-actor spec (matches scripts/run_setup1_target.cmd).
SMOKE = False   # <<< FLIP to True for a quick Colab/Kaggle smoke; False = full run

# ---- model size (full spec: width 512, 6 residual blocks) -------------------
HIDDEN          = 64  if SMOKE else 512    # d: trunk width  ("model dim")
N_RES_BLOCKS    = 2   if SMOKE else 6      # residual MLP blocks in the TRUNK (pre-LN ResNet)
# ---- trunk architecture (configurable: stacked transformer w/ ResNet-MLP ends) ----
# CPU-probe sizing: HIDDEN=256 + N_TX_LAYERS=4 ~ same fwd cost as the old h=512 trunk; h=512,L=4 ~3x.
ARCH         = "transformer"  # "trunk" = legacy (1x attn + N res-MLP) | "transformer" = N-layer stack
N_TX_LAYERS  = 4              # [transformer] encoder layers, each = multi-head attn + ResNet-MLP
N_HEADS      = 8              # [transformer] attention heads (HIDDEN must be divisible by this)
TX_MLP_RATIO = 4              # [transformer] per-layer MLP hidden = ratio x HIDDEN
N_STEM_RES   = 1              # [transformer] ResNet-MLP blocks BEFORE the stack (input end)
N_HEAD_RES   = 1              # [transformer] ResNet-MLP blocks AFTER  the stack (output end)
VALUE_RES_BLOCKS = 1  if SMOKE else 2      # residual MLP blocks in the VALUE/critic head (same ResNet)
D_G             = 32                        # board-globals embedding dim
USE_GLU         = True
USE_ATTENTION   = True                      # cross-planet self-attention block (A/B: False = ablate it)
STD_STATE_DEP   = True                      # state-dependent phi log-std head

# ---- env / batch (B = NUM_GROUPS * GROUP_SIZE; PPO has no groups, B = env count)
NUM_GROUPS      = 4   if SMOKE else 16
GROUP_SIZE      = 4   if SMOKE else 8
B               = NUM_GROUPS * GROUP_SIZE   # parallel envs (= 16 smoke / 128 full)
EPISODE_STEPS   = 200 if SMOKE else 500
PLANET_CAP      = 48                         # E: planet slots/env == dest-categorical size (comets omitted)
FLEET_CAP       = 512 if SMOKE else 1024     # max simultaneous in-flight fleets/env
ACT_THRESHOLD   = 0.05                       # tau_act: commit when phi >= this
SHIP_SPEED      = 6.0
COMETS_ENABLED  = True                       # spawn comets in training (hidden schedule, like the official engine)
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450) # ticks at which a symmetric comet pair appears
COMET_SPEED     = 4.0                         # comet straight-line speed (official cometSpeed)
COMET_PAIRS     = 1                           # point-symmetric comet pairs per spawn (-> 2 comets)
COMET_SPAWN_RADIUS = 50.0                     # radius from center where comets enter
COMET_PERI_MIN  = 15.0                        # comet closest-approach to center (min)
COMET_PERI_MAX  = 30.0                        # comet closest-approach to center (max)
COMET_EXPIRE_RADIUS = 56.0                    # comet removed (ships lost) once it sweeps back past this

# ---- PPO + GAE --------------------------------------------------------------
LR              = 1e-4
GAMMA           = 0.99
GAE_LAMBDA      = 0.95
CLIP            = 0.2
VF_COEF         = 0.5
ENT_COEF        = 0.005
MINIBATCHES     = 16  if SMOKE else 64
UPDATE_EPOCHS   = 1
MAX_GRAD_NORM   = 0.5
KL_TARGET       = 0.05  # PPO guardrail: stop the update epoch once a minibatch's KL exceeds this (anti-runaway)
LOGRATIO_CLAMP  = 4.0   # PPO guardrail: clamp log importance-ratio before exp (ratio in [e^-4, e^4]; no overflow)
LOGIT_CLAMP     = 8.0   # Dirichlet guardrail: clamp dest_logits before softplus so alpha can't explode
ADAM_EPS        = 1e-5

# ---- logstd cap anneal: 0 -> -1.2 over the first 100 iters, then -1.0 --------
LOGSTD_MIN      = -2.0
LOGSTD_MAX      = 0.0      # phase-1 start
LOGSTD_MAX_END  = -1.2     # phase-1 forced-decay target
LOGSTD_MAX_POST = -1.0     # phase-2 cap (head takes over)
SIGMA_DECAY_ITERS = 100    # phase-1 length in iterations

# ---- calm init --------------------------------------------------------------
INIT_MU_SCALE   = 0.02     # small final phi-mu-layer weights
INIT_PHI_BIAS   = -4.0     # negative phi bias (few commits at init)

# ---- training length (NO hand-coded stages -- the Elo/PFSP league IS the curriculum) ----
# Every iter the opponent is PFSP-sampled from the league {random, starter, snapshots} by Elo,
# so the schedule EMERGES from skill: a fresh learner (rated at the random anchor) plays the weak
# scripted bots, and as its rating climbs it shifts to ever-stronger historical snapshots. The old
# stage1/2/3 boundaries are gone -- the Elo ladder reproduces them organically.
TOTAL_ITERS       = 4 if SMOKE else 1400
SELFPLAY_REFRESH  = 5 if SMOKE else 100   # snapshot the learner into the pool every N iters
SNAPSHOT_WARMUP   = 0 if SMOKE else 100   # anchors-only warmup: no learner snapshot before this iter

# ---- self-play Elo LEAGUE (PFSP pool -- the SOLE opponent source, no stages) -----
# EVERY iter the opponent is PFSP-sampled from ALL pool members (random + starter anchors +
# learner snapshots), weighted by f(p)+floor where p is the Elo-expected score. The two scripted
# bots are rating-ANCHORED (fixed Elo) so the scale -- and the auto-curriculum -- are pinned.
LEAGUE_MAX_SNAPSHOTS = 4 if SMOKE else 12   # max AUTO learner snapshots kept (FIFO; pinned ckpts exempt)
LEAGUE_INIT_CKPTS    = []     # paths to past .pt weights to seed the pool (pinned, never pruned)
ELO_INIT             = 0.0    # learner starts at the random anchor; climbs the ladder as it improves
ELO_RANDOM           = 0.0    # ANCHORED (fixed) rating of the random bot  -> pins the Elo scale
ELO_STARTER          = 600.0  # ANCHORED rating of the starter bot (always-on grounding baseline)
ELO_MEDIUM           = 1000.0  # MEDIUM bot (starter++; advanced; locked until the advance gate)
ELO_GREEDY           = 900.0   # GREEDY local-capture bot (advanced; locked until the advance gate)
ELO_INTERMEDIATE     = 750.0   # INTERMEDIATE bot (in-flight-aware capture + counter + rebalance + comet-escape); ALWAYS-ON, not gated
ADVANCE_TRIGGER_ELO  = 800.0   # learner ELO that triggers an all-anchor recalibration to consider unlocking
ADVANCE_CONFIRM_ELO  = 700.0   # if the all-anchor recal puts the learner below this, stay in base regime (anti-inflation)
ELO_K                = 32.0   # Elo K-factor (bumped: Elo now DRIVES sampling, so track skill fast)
PFSP_MODE            = "even" # snapshot sampling: "even" (p*(1-p)) / "hard" ((1-p)^P) / "uniform"
PFSP_POWER           = 2.0    # exponent when PFSP_MODE == "hard"
PFSP_FLOOR           = 0.05   # uniform floor added to every snapshot weight (anti-forgetting)
MATCH_ELO_W          = 0.7    # matchmaking weight on Elo (evenly-matched opponents)
MATCH_FP_W           = 0.3    # matchmaking weight on behavioral-footprint distance (anti-clone)
SCRIPT_MATCH_BOOST   = 2.0    # x weight on UNMASTERED scripted anchors (2.0 = +100%); spend compute on the bots until beaten
SCRIPT_BOOST_DROP_WR = 0.90   # drop a scripted anchor's boost once learner EMA win-rate vs it >= this
SCRIPT_BOOST_MIN_GAMES = 5    # min matches vs an anchor before its boost can be dropped
WR_EMA_BETA          = 0.1    # EMA smoothing for per-opponent learner win-rate (responsive 90% gate)
CAPTURE_RADIUS       = 25.0   # greedy bot: only captures non-owned planets within this distance
CAPTURE_MARGIN       = 2.0    # greedy/intermediate: ships sent = floor(target_ships) + this (transit-production buffer)
REBALANCE_PCT        = 0.4    # intermediate: rebalance when a source's surplus over the local-poorest > this fraction of garrison
REBALANCE_RADIUS     = 30.0   # intermediate: only rebalance to owned planets within this distance (local, efficient)

# ---- DISCRETE action space + sun-reachability (AlphaZero-style, tree-searchable) -----
DISCRETE_PHI = True           # phi = launch-fraction CATEGORICAL (vs squashed-Gaussian) -> tree search
PHI_STEP     = 0.05           # bucket granularity -> minimum non-zero dispatch = 5% of a planet's ships
PHI_BINS     = [0.0] + [round(PHI_STEP * k, 6) for k in range(1, int(round(1.0 / PHI_STEP)) + 1)]
N_PHI        = len(PHI_BINS)  # 21 buckets: noop(0%) + 5%,10%,...,100%
PHI_BINS_T   = torch.tensor(PHI_BINS, dtype=DTYPE, device=DEVICE)
REACH_MASK   = True           # hard-mask destinations the sun would absorb (exact for straight shots)
LEAD_TARGET  = True           # aim at the orbiting destination's INTERCEPT, not its current position
LAUNCHES_PER_PLANET = 10      # (legacy K-slot knob; unused by the Dirichlet allocation actor)
# ---- Dirichlet allocation action space (per-source softmax over dests; self-diagonal = hold) ----
ALLOC_KAPPA      = 1.0   # Dirichlet concentration scale: alpha = softplus(logits)*kappa + eps
MIN_LAUNCH_SHIPS = 2     # drop launches below this many ships (-> stay home; no 1-ship dribbles)
POLICY_MODE      = "ppo" # "ppo" = Dirichlet PPO | "mcts" = discretized tree search (stub)
LAUNCH_REWARD    = 0.001 # dispatch reward: factor on ships launched to NON-ego planets (attack/expand), per step. Tune vs production (mean ~2.7/planet, ~76/world)
SELF_LAUNCH_REWARD = 0.001 # dispatch reward: factor on ships launched to EGO-owned planets (reinforce), per step; tunable (<= LAUNCH_REWARD to favor attacks)
SELF_LAUNCH_CAP  = 10.0  # CAP 1: max EGO-dest ships/step counted in the dispatch reward (non-ego launches UNCAPPED)
LAUNCH_STEP_CAP  = 0.5   # CAP 2: overall per-step cap on the dispatch reward (enemy*f + capped self*f, combined)
LAUNCH_WINDOW    = 150   # launch reward VALID only for the first N steps; a WIN before step N pays the unspent window IN FULL (anti-stall cashout)
LAUNCH_GAME_CAP  = 30.0  # CAP 3: cap on TOTAL launch reward over the whole game (per-step + cashout); window-max = LAUNCH_STEP_CAP*LAUNCH_WINDOW

# ---- reward (v5, docs/set-ups/1.md -- REPLACES the old event reward) --------
WIN_BONUS         = 1000.0   # win outcome
LOSS_PENALTY      = 1000.0   # loss outcome (symmetric)
WIN_DECAY         = 0.995    # win value  = +WIN_BONUS   * WIN_DECAY^max(0, len - DECAY_START)
LOSS_DECAY        = 0.995    # loss value = -LOSS_PENALTY* LOSS_DECAY^max(0, len - DECAY_START)
DECAY_START_STEP  = 100.0    # win/loss FLAT for the first 100 steps, then decay
CAPTURE_REWARD    = 30.0     # + per planet gained (per ego owner-flip, per step)
CAPTURE_LOSS_FRAC = 0.9      # losing a planet removes only this fraction of CAPTURE_REWARD (net +10%/capture)
CAPTURE_PROD_SCALE = 0.2     # per-planet capture reward = CAPTURE_REWARD * (1 + this * production); higher = grab valuable planets
PROD_MILESTONE_REWARD = 20.0 # +20 each time ego TOTAL ships crosses a doubling threshold
PROD_MILESTONE_BASE   = 100.0
PPO_REWARD_SCALE  = 100.0    # divide per-step reward AND outcome by this before GAE (value stability)

# ---- checkpointing (Colab/Kaggle-friendly path) -----------------------------
import os
CKPT_DIR = os.environ.get("OW_CKPT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working")
                          else "/content" if os.path.isdir("/content") else ".")
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "setup1_target_policy.pt")
BEST_CKPT_PATH = os.path.join(CKPT_DIR, "setup1_target_policy_best.pt")
TRAIN_STATE_PATH = os.path.join(CKPT_DIR, "setup1_target_train_state.pt")  # full resumable state
CKPT_EVERY = 100  # save cadence (iters); best-by-win-rate is saved separately

# ---- world pool -------------------------------------------------------------
N_WORLDS        = 256 if SMOKE else 2048
SEED            = 0
WORLD_RESAMPLE_EVERY = 0 if SMOKE else 100    # regenerate the world pool every N GLOBAL iters (0 = fixed pool)
ELO_RECAL_EVERY      = 0 if SMOKE else 1000   # re-ground league ELO vs anchors every N GLOBAL iters (decoupled from reseed)
RESUME_FROM          = None                    # set to TRAIN_STATE_PATH (or a path) to resume + train more
ELO_RECAL_ENVS       = 64                      # games vs each fixed anchor when re-grounding ELO at a resample boundary

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("SMOKE =", SMOKE, "| B =", B, "| HIDDEN =", HIDDEN,
      "| N_RES_BLOCKS =", N_RES_BLOCKS, "| TOTAL_ITERS =", TOTAL_ITERS)
print("checkpoints ->", CKPT_PATH)

## 3. Feature layout (mirrors `core/encode.hpp` + `core/state.hpp`)

`F = 11 body + 90 threat = 101` features/planet; `G = 10` board globals (a separate input).
Board constants are the competition's. The threat features keep the `N_SOON=15` soonest +
`N_BIG=15` largest inbound fleets per planet, 3 feats each.

In [ ]:
# board constants (core/state.hpp)
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PI = math.pi

# feature layout (core/encode.hpp)
N_SOON = 15
N_BIG = 15
N_THREAT_FLEETS = N_SOON + N_BIG            # 30 slots * 3 feats = 90
N_BODY_FEATURES = 11
N_ENTITY_FEATURES = N_BODY_FEATURES + 3 * N_THREAT_FLEETS   # 32
N_GLOBAL_FEATURES = 10
F_DIM = N_ENTITY_FEATURES
G_DIM = N_GLOBAL_FEATURES

SHIP_LOG_DENOM = math.log(1000.0)
DIAG_HALF = math.sqrt(BOARD_SIZE * BOARD_SIZE + BOARD_SIZE * BOARD_SIZE) / 2.0
THREAT_MAX_SPEED = 6.0      # == ship speed; fleet speed cap for the ETA model
THREAT_ETA_SCALE = 100.0
BIG = 1e18

def ship_log_t(x):
    return torch.log1p(x.clamp_min(0.0)) / SHIP_LOG_DENOM

def fleet_speed_t(ships, vmax=SHIP_SPEED):
    # 1 + (vmax-1)*(log(ships)/log(1000))^1.5, capped, ships>=1  (core/encode.hpp)
    n = ships.clamp_min(1.0)
    v = 1.0 + (vmax - 1.0) * torch.pow(torch.log(n) / math.log(1000.0), 1.5)
    return v.clamp_max(vmax)

print("F =", F_DIM, "| G =", G_DIM, "| dest-categorical size E =", PLANET_CAP, "| action params/planet = 2 (dest, phi)")

## 4. Target-actor distribution (mirrors `model/distribution.hpp::TargetActorDist`)
Per OWNED planet: a destination Categorical over the E obs slots (masked to live
planets) x a squashed-Gaussian phi (% of that planet's ships). Joint log-prob / entropy
sum over owned planets only.

In [ ]:
_LOG2PI = 1.8378770664093453        # log(2*pi)
_HALF_LOG2PIE = 1.4189385332046727  # 0.5*log(2*pi*e)


class TargetActorDist:
    '''v5 TARGET actor distribution (mirrors model/distribution.hpp::TargetActorDist).

    Per OWNED planet the action is (dest, phi):
      - dest ~ Categorical over the E obs slots, MASKED to LIVE planets (dead-destination
        logits set to -1e9 before softmax).
      - phi  = sigmoid(N(mu, sigma^2)) launch fraction (squashed Gaussian, same as the old phi).
    Joint log-prob / entropy SUM over owned planets only (unowned -> forced no-op, dropped).
    Stored action is (B,E,2): [...,0]=dest row index (float), [...,1]=phi in (0,1).

    Inputs:
      dest_logits (B,E,E); phi_mean/phi_logstd (B,E,1) [logstd pre-clipped];
      owned (B,E) = legal source planets; alive (B,E) = valid destination slots.
    '''
    def __init__(self, dest_logits, phi_mean, phi_logstd, owned, alive, reach=None):
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)        # (B,E)
        self.pmean = phi_mean.squeeze(-1)                                    # (B,E)
        self.plogstd = phi_logstd.squeeze(-1)                               # (B,E)
        self.pstd = torch.exp(self.plogstd)                                # (B,E)
        dead_dest = (alive < 0.5).unsqueeze(1)                             # (B,1,E) over the source dim
        if reach is not None:                                             # also forbid sun-absorbed dests
            dead_dest = dead_dest | (reach < 0.5)                          # (B,E,E) per-source reachability
        self.ml = dest_logits.masked_fill(dead_dest, -1e9)                 # (B,E,E) live & reachable dests
        self.dlogp = torch.log_softmax(self.ml, -1)                        # (B,E,E)

    def _mask_action(self, a):                  # zero out unowned planets (-> no-op, phi=0)
        return a * self.owned.unsqueeze(-1)

    def sample(self):
        B, E = self.ml.shape[0], self.ml.shape[1]
        p = torch.softmax(self.ml, -1).reshape(B * E, E)
        dest = torch.multinomial(p, 1).reshape(B, E).to(DTYPE)             # (B,E)
        phi = torch.sigmoid(self.pmean + self.pstd * torch.randn_like(self.pmean))  # (B,E)
        return self._mask_action(torch.stack([dest, phi], -1))            # (B,E,2)

    def greedy(self):
        dest = self.ml.argmax(-1).to(DTYPE)                               # (B,E)
        phi = torch.sigmoid(self.pmean)                                   # (B,E)
        return self._mask_action(torch.stack([dest, phi], -1))

    def log_prob(self, a):
        E = self.ml.shape[1]
        dest = a[..., 0].to(torch.long).clamp(0, E - 1)                   # (B,E)
        dest_lp = self.dlogp.gather(2, dest.unsqueeze(-1)).squeeze(-1)    # (B,E)
        phi = a[..., 1]                                                   # (B,E)
        ac = phi.clamp(1e-6, 1.0 - 1e-6)
        z = torch.log(ac) - torch.log1p(-ac)                             # logit(phi) recovers latent z
        mu = self.pmean.clamp(-15.0, 15.0)
        logN = -0.5 * (z - mu).pow(2) / (self.pstd * self.pstd) - self.plogstd - 0.5 * _LOG2PI
        logjac = torch.log(ac) + torch.log1p(-ac)                        # log|da/dz| = log(a(1-a))
        phi_lp = logN - logjac                                           # (B,E)
        return ((dest_lp + phi_lp) * self.owned).sum(1)                  # (B,) owned-masked sum

    def entropy(self):
        cat_e = -(torch.softmax(self.ml, -1) * self.dlogp).sum(-1)       # (B,E) categorical entropy
        phi_e = self.plogstd + _HALF_LOG2PIE                            # (B,E) phi latent-Gaussian entropy
        return ((cat_e + phi_e) * self.owned).sum(1)                    # (B,)

class DiscreteTargetDist:
    """Per-source Dirichlet ALLOCATION over the E destinations (self-diagonal = hold/no-launch).
    dest_logits (B,E,E) -> alpha = softplus(logits)*kappa + eps -> Dirichlet rows. The action is the
    allocation matrix A (B,E,E), each row a point on the E-simplex. Dead/unreachable dests and the
    self-diagonal are handled at DECODE (ships->0->held), so the Dirichlet itself stays unmasked &
    numerically clean. Joint log-prob/entropy SUM over owned source planets."""
    def __init__(self, dest_logits, owned, alive=None, reach=None, ships=None, kappa=ALLOC_KAPPA):
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)        # (B,E)
        self.alpha = F.softplus(dest_logits.clamp(-LOGIT_CLAMP, LOGIT_CLAMP)) * kappa + 1e-3   # (B,E,E) concentration (clamped -> can't explode)
        self.dist = torch.distributions.Dirichlet(self.alpha)
    def _mask_action(self, a):
        return a                                                            # decode enforces legality
    def sample(self):
        return self.dist.rsample()                                          # (B,E,E) allocation rows ~ simplex
    def greedy(self):
        return self.alpha / self.alpha.sum(-1, keepdim=True)                # mean allocation
    def log_prob(self, a):
        a = a.clamp_min(1e-6); a = a / a.sum(-1, keepdim=True)              # project to the simplex (safety)
        return (self.dist.log_prob(a) * self.owned).sum(1)                  # (B,)
    def entropy(self):
        return (self.dist.entropy() * self.owned).sum(1)                    # (B,)

## 5. World generator (pure Python mirror of official `generate_planets`)

Mirrors `REFERENCE_orbit_wars.py::generate_planets` + `native_worldgen.generate_world`:
4-fold symmetric planet groups (each `[id, owner, x, y, radius, ships, production]`), a
guaranteed `MIN_STATIC_GROUPS=3` static groups via polar sampling, `MIN/MAX_PLANET_GROUPS`
total, then home assignment (`base` group: planet 0 -> player 0 with 10 ships, planet 3 ->
player 1 with 10 ships). `angular_velocity ~ U(0.025, 0.05)`.

**Simplification:** no comet schedule (comets omitted). Worlds are packed straight into the
batched env tensors (no `.owp` files).

In [ ]:
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
PLANET_CLEARANCE = 7

def _dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def generate_planets(rng):
    '''Faithful mirror of REFERENCE_orbit_wars.py::generate_planets.
    Returns rows [id, owner, x, y, radius, ships, production].'''
    planets = []
    num_q1 = rng.randint(MIN_PLANET_GROUPS, MAX_PLANET_GROUPS)
    idc = 0
    # Phase 1: guaranteed static groups (polar sampling).
    static_groups = 0
    for _ in range(5000):
        if static_groups >= MIN_STATIC_GROUPS:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        angle = rng.uniform(0, math.pi / 2)
        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orbital > max_orbital:
            continue
        orbital_r = rng.uniform(min_orbital, max_orbital)
        x = CENTER + orbital_r * math.cos(angle)
        y = CENTER + orbital_r * math.sin(angle)
        if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
            continue
        if (BOARD_SIZE - x) - r < 0 or (BOARD_SIZE - y) - r < 0:
            continue
        if (x - CENTER) < r + 5 or (y - CENTER) < r + 5:
            continue
        ships = min(rng.randint(5, 99), rng.randint(5, 99))
        # NOTE: the reference stores rows as [id, owner, y, x, r, ...] (x/y swapped naming),
        # which is just a relabel of the symmetric copies; we keep it identical.
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        valid = True
        for tp in tps:
            for p in planets:
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
            if not valid:
                break
        if valid:
            planets.extend(tps); idc += 4; static_groups += 1
    # Phase 2: fill remaining groups (normal random loop).
    attempts = 0
    max_attempts = 5000
    has_orbiting = False
    while len(planets) < num_q1 * 4 or (not has_orbiting and attempts < max_attempts):
        attempts += 1
        if attempts >= max_attempts:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        x = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        y = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        orbital_radius = _dist((x, y), (CENTER, CENTER))
        if orbital_radius < SUN_RADIUS + r + 10:
            continue
        if orbital_radius + r >= ROTATION_RADIUS_LIMIT:
            if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
                continue
        valid = True
        ships = rng.randint(5, 30)
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        for tp in tps:
            tp_orb = _dist((tp[2], tp[3]), (CENTER, CENTER))
            tp_rot = tp_orb + tp[4] < ROTATION_RADIUS_LIMIT
            for p in planets:
                p_orb = _dist((p[2], p[3]), (CENTER, CENTER))
                p_rot = p_orb + p[4] < ROTATION_RADIUS_LIMIT
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
                if tp_rot != p_rot:
                    if abs(tp_orb - p_orb) < tp[4] + p[4] + PLANET_CLEARANCE:
                        valid = False; break
            if not valid:
                break
        if valid:
            if orbital_radius + r < ROTATION_RADIUS_LIMIT:
                has_orbiting = True
            planets.extend(tps); idc += 4
    return planets

def generate_world(seed):
    '''Mirror of native_worldgen.generate_world (comets dropped).'''
    rng = random.Random(seed)
    angular_velocity = rng.uniform(0.025, 0.05)
    planets = generate_planets(rng)
    num_groups = len(planets) // 4
    if num_groups > 0:
        base = rng.randint(0, num_groups - 1) * 4
        planets[base][1] = 0;      planets[base][5] = 10       # player 0 home
        planets[base + 3][1] = 1;  planets[base + 3][5] = 10   # player 1 home
    return {"planets": planets, "angular_velocity": angular_velocity}

def make_world_pool(n, base_seed=0):
    return [generate_world(base_seed + i) for i in range(n)]

# quick sanity: one world's shape
_w = generate_world(0)
print("world 0:", len(_w["planets"]), "planets | ang_vel = %.4f" % _w["angular_velocity"])

## 6. Batched env (mirrors `rl/gpu_env.cpp`)

Structure-of-Arrays over `B` envs, leading dim `B`. Integer quantities (ships, production,
owner, step) live in float32 (exact below `2^24`, cheaper on GPU). Planets in fixed slots
`[0, PLANET_CAP)`; fleets in a fixed pool `[0, FLEET_CAP)` with an alive mask. **No comet
slots** (comets omitted).

`reset` packs a list of world dicts into the device tensors and zeroes runtime state.

In [ ]:
class GpuEnv:
    def __init__(self, planet_cap, fleet_cap, episode_steps, ship_speed, device):
        self.Ec = planet_cap
        self.Fc = fleet_cap
        self.T = episode_steps
        self.vmax = ship_speed
        self.dev = device
        self.B = 0

    def reset(self, worlds):
        B = len(worlds)
        self.B = B
        Ec, Fc = self.Ec, self.Fc
        dev = self.dev
        z = lambda *s: torch.zeros(s, dtype=DTYPE, device=dev)
        # planets (B, Ec)
        self.p_alive = z(B, Ec)
        self.p_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
        self.p_x = z(B, Ec); self.p_y = z(B, Ec); self.p_radius = z(B, Ec)
        self.p_ships = z(B, Ec); self.p_prod = z(B, Ec); self.p_is_comet = z(B, Ec)
        self.p_init_x = z(B, Ec); self.p_init_y = z(B, Ec); self.p_rotates = z(B, Ec)
        self.p_comet_vx = z(B, Ec); self.p_comet_vy = z(B, Ec)   # comet straight-line velocity
        # fill from worlds (CPU numpy then upload once)
        pa = np.zeros((B, Ec), np.float32); po = np.full((B, Ec), -1.0, np.float32)
        px = np.zeros((B, Ec), np.float32); py = np.zeros((B, Ec), np.float32)
        pr = np.zeros((B, Ec), np.float32); ps = np.zeros((B, Ec), np.float32)
        pp = np.zeros((B, Ec), np.float32); pix = np.zeros((B, Ec), np.float32)
        piy = np.zeros((B, Ec), np.float32); prot = np.zeros((B, Ec), np.float32)
        angv = np.zeros((B,), np.float32)
        for b, w in enumerate(worlds):
            pls = w["planets"][:Ec]
            for i, pl in enumerate(pls):
                pid, owner, x, y, rad, sh, prod = pl
                pa[b, i] = 1.0; po[b, i] = float(owner)
                px[b, i] = x; py[b, i] = y; pr[b, i] = rad
                ps[b, i] = sh; pp[b, i] = prod
                pix[b, i] = x; piy[b, i] = y
                rr = math.hypot(x - CENTER, y - CENTER)
                prot[b, i] = 1.0 if (rr + rad < ROTATION_RADIUS_LIMIT) else 0.0
            angv[b] = w["angular_velocity"]
        t = lambda a: torch.from_numpy(a).to(dev)
        self.p_alive = t(pa); self.p_owner = t(po); self.p_x = t(px); self.p_y = t(py)
        self.p_radius = t(pr); self.p_ships = t(ps); self.p_prod = t(pp)
        self.p_init_x = t(pix); self.p_init_y = t(piy); self.p_rotates = t(prot)
        self.p_is_comet = z(B, Ec)
        # fleets (B, Fc)
        self.f_alive = z(B, Fc); self.f_owner = z(B, Fc); self.f_x = z(B, Fc)
        self.f_y = z(B, Fc); self.f_angle = z(B, Fc); self.f_ships = z(B, Fc)
        self.f_seq = z(B, Fc)
        # per-env
        self.ang_vel = t(angv)
        self.step_ct = torch.zeros(B, dtype=DTYPE, device=dev)
        self.done = torch.zeros(B, dtype=DTYPE, device=dev)

## 7. Env -- `fleet_target_batch` + `encode`

`fleet_target_batch`: closed-form ray-vs-disk + sun occlusion for `M` virtual fleets/env vs
the current planets (mirrors `gpu_env.cpp::fleet_target_batch`). Used by `encode` (threat
features, `M=Fc`) and by the valid-launch check.

`encode(ego)`: the batched mirror of `encode_obs`. Planets stay in fixed slots (the per-planet
actor is permutation-equivariant). Returns entities `(B,Ec,F)`, entity_mask, action_mask,
globals `(B,Ec...)`.

In [ ]:
def fleet_target_batch(env, fx, fy, fang, fships, vmax):
    '''fx,fy,fang,fships: (B,M). Returns tgt (B,M) long (planet slot or -1), eta (B,M).'''
    dx = torch.cos(fang).unsqueeze(2)           # (B,M,1)
    dy = torch.sin(fang).unsqueeze(2)
    pxr = env.p_x.unsqueeze(1)                  # (B,1,Ec)
    pyr = env.p_y.unsqueeze(1)
    rr = (env.p_radius * env.p_radius).unsqueeze(1)
    alive = (env.p_alive > 0.5).unsqueeze(1)
    ox = fx.unsqueeze(2) - pxr                  # (B,M,Ec)
    oy = fy.unsqueeze(2) - pyr
    tca = -(ox * dx + oy * dy)
    perp2 = ox * ox + oy * oy - tca * tca
    hit = (tca >= 0.0) & (perp2 <= rr) & alive
    t_int = (tca - torch.sqrt((rr - perp2).clamp_min(0.0))).clamp_min(0.0)
    tvals = torch.where(hit, t_int, torch.full_like(t_int, BIG))
    best_t, best_e = tvals.min(2)               # (B,M)
    any_hit = best_t < BIG
    # sun occlusion
    sx = fx - CENTER; sy = fy - CENTER
    dxx = torch.cos(fang); dyy = torch.sin(fang)
    tcs = -(sx * dxx + sy * dyy)
    sperp2 = sx * sx + sy * sy - tcs * tcs
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = tcs - torch.sqrt((sr - sperp2).clamp_min(0.0))
    sun_block = (tcs >= 0.0) & (sperp2 <= sr) & (t_sun >= 0.0) & (t_sun < best_t)
    valid = any_hit & (~sun_block)
    tgt = torch.where(valid, best_e, torch.full_like(best_e, -1))
    eta = best_t / fleet_speed_t(fships, vmax)
    eta = torch.where(valid, eta, torch.zeros_like(eta))
    return tgt, eta


def env_encode(env, ego=0):
    B, Ec, Fc = env.B, env.Ec, env.Fc
    enemy = 1 - ego
    na = 2
    dev = env.dev
    alive = env.p_alive > 0.5
    av = env.p_alive
    owner = env.p_owner
    dxc = env.p_x - CENTER; dyc = env.p_y - CENTER
    dist = torch.sqrt(dxc * dxc + dyc * dyc)
    comet = env.p_is_comet > 0.5
    rotating = (~comet) & ((dist + env.p_radius) < ROTATION_RADIUS_LIMIT)
    angv = env.ang_vel.unsqueeze(1)
    vmag = torch.where(rotating, angv.abs() * dist, torch.zeros_like(dist))
    cw = torch.where(rotating,
                     torch.where(angv >= 0.0, torch.ones_like(dist), -torch.ones_like(dist)),
                     torch.zeros_like(dist))
    # ego-relative ownership code: 0 neutral, 1 ego, 2..na enemy
    m = owner - float(ego) + float(na)
    m = m - float(na) * torch.floor(m / float(na))
    own_code = torch.where(owner < 0.0, torch.zeros_like(owner), 1.0 + m)
    b0 = env.p_x / BOARD_SIZE
    b1 = env.p_y / BOARD_SIZE
    b2 = vmag / THREAT_MAX_SPEED
    b3 = cw
    b4 = env.p_radius / 3.0
    b5 = env.p_prod / 5.0
    b6 = ship_log_t(env.p_ships)
    b7 = own_code
    b8 = dist / DIAG_HALF
    b9 = comet.to(DTYPE)
    actable = (owner == float(ego)) & alive & (env.p_ships > 0.0)
    b10 = actable.to(DTYPE)
    body = torch.stack([b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10], 2)  # (B,Ec,11)

    # threat: per planet, N_SOON soonest + N_BIG largest inbound fleets
    ftgt, feta = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
    falive = env.f_alive > 0.5
    fsign = torch.where(env.f_owner == float(ego), torch.ones_like(env.f_ships),
                        -torch.ones_like(env.f_ships))
    fslog = ship_log_t(env.f_ships.abs())
    slot = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, Ec, 1)
    tgtf = ftgt.to(DTYPE).unsqueeze(1)                       # (B,1,Fc)
    targeting = (tgtf == slot) & (ftgt >= 0).unsqueeze(1) & falive.unsqueeze(1)  # (B,Ec,Fc)
    W = float(1 << 20)
    eta_mat = torch.where(targeting, feta.unsqueeze(1), torch.full((1,), BIG, device=dev))
    absf = env.f_ships.abs().unsqueeze(1)
    seqf = env.f_seq.unsqueeze(1)
    key_big = torch.where(targeting, absf * W - seqf, torch.full((1,), -BIG, device=dev))
    eta_top_v, eta_top_i = torch.topk(eta_mat, N_SOON, 2, largest=False)
    big_top_v, big_top_i = torch.topk(key_big, N_BIG, 2, largest=True)
    idx = torch.cat([eta_top_i, big_top_i], 2)              # (B,Ec,7)
    valid = torch.cat([eta_top_v < BIG * 0.5, big_top_v > -BIG * 0.5], 2)
    def pick(src):
        return src.unsqueeze(1).expand(B, Ec, Fc).gather(2, idx)
    zt = torch.zeros(B, Ec, N_THREAT_FLEETS, device=dev)
    sgn = torch.where(valid, pick(fsign), zt)
    etf = torch.where(valid, pick(feta) / THREAT_ETA_SCALE, zt)
    slg = torch.where(valid, pick(fslog), zt)
    threat = torch.stack([sgn, etf, slg], 3).reshape(B, Ec, 3 * N_THREAT_FLEETS)  # (B,Ec,21)

    entities = torch.cat([body, threat], 2) * av.unsqueeze(2)   # zero dead slots
    entity_mask = av
    action_mask = b10

    # globals (B,10)
    def psum(mask):
        return (env.p_ships * mask.to(DTYPE)).sum(1)
    mine = (owner == float(ego)) & alive
    en = (owner == float(enemy)) & alive
    neu = (owner < 0.0) & alive
    my_ships = psum(mine); en_ships = psum(en)
    my_pl = mine.to(DTYPE).sum(1); en_pl = en.to(DTYPE).sum(1); neu_pl = neu.to(DTYPE).sum(1)
    npl = av.sum(1); total = npl.clamp_min(1.0)
    my_fleet = (env.f_ships * (env.f_owner == float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    en_fleet = (env.f_ships * (env.f_owner == float(enemy)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    ME = 40.0
    g0 = env.step_ct / max(1, env.T)
    g1 = env.ang_vel * 10.0
    g2 = ship_log_t(my_ships); g3 = ship_log_t(en_ships)
    g4 = my_pl / total; g5 = en_pl / total; g6 = neu_pl / total
    g7 = ship_log_t(my_fleet); g8 = ship_log_t(en_fleet)
    g9 = torch.minimum(npl, torch.full_like(npl, ME)) / ME
    globals_ = torch.stack([g0, g1, g2, g3, g4, g5, g6, g7, g8, g9], 1)
    return entities, entity_mask, action_mask, globals_

## 8. Env -- `launch_fleets` + scripted opponents

`launch_fleets`: scatter committed launches into free fleet slots (mirrors
`gpu_env.cpp::launch_fleets`; ship deduction happens in `step`). `opponent_action`: noop /
random / starter (mirrors `gpu_env.cpp::opponent_action`).

In [ ]:
def launch_fleets(env, owner, from_slot, angle, ships, commit, seq):
    '''Append committed launches (all (B,L)) into the fleet pool, distinct free slots.'''
    B, Fc = env.B, env.Fc
    dev = env.dev
    L = from_slot.shape[1]
    free = env.f_alive < 0.5
    freef = free.to(DTYPE)
    fr = torch.cumsum(freef, 1) - 1.0
    n_free = freef.sum(1)
    idx = torch.where(free, fr.long(), torch.full_like(fr.long(), Fc))
    rank_to_slot = torch.full((B, Fc + 1), Fc, dtype=torch.long, device=dev)
    slot_src = torch.arange(Fc, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Fc)
    rank_to_slot.scatter_(1, idx, slot_src)
    rank_to_slot = rank_to_slot[:, :Fc]
    commitb = commit > 0.5
    crank = (torch.cumsum(commit.to(DTYPE), 1) - 1.0).long()
    place = commitb & (crank < n_free.unsqueeze(1).long())
    gslot = rank_to_slot.gather(1, crank.clamp(0, Fc - 1))
    dump = torch.full_like(gslot, Fc)
    wslot = torch.where(place, gslot, dump)
    def scatter_into(field, vals):
        aug = torch.cat([field, torch.zeros(B, 1, dtype=DTYPE, device=dev)], 1)
        aug.scatter_(1, wslot, vals)
        return aug[:, :Fc]
    fs = from_slot.clamp(0, env.p_x.shape[1] - 1)
    opx = env.p_x.gather(1, fs); opy = env.p_y.gather(1, fs); orad = env.p_radius.gather(1, fs)
    sx = opx + torch.cos(angle) * (orad + 0.1)
    sy = opy + torch.sin(angle) * (orad + 0.1)
    onesL = torch.ones(B, L, dtype=DTYPE, device=dev)
    env.f_alive = scatter_into(env.f_alive, onesL)
    env.f_owner = scatter_into(env.f_owner, owner)
    env.f_x = scatter_into(env.f_x, sx)
    env.f_y = scatter_into(env.f_y, sy)
    env.f_angle = scatter_into(env.f_angle, angle)
    env.f_ships = scatter_into(env.f_ships, ships)
    env.f_seq = scatter_into(env.f_seq, seq)


def opponent_action(env, opponent):
    '''Scripted player-1 launches: one per owned planet, half garrison (>=20).
    0=random heading, 1=starter (nearest static non-owned), 2=noop, 3=medium (starter++: nearest incl. rotating, lead-aim),
    4=greedy (nearest beatable non-owned planet within CAPTURE_RADIUS, just-enough ships),
    5=intermediate (in-flight-aware capture from nearest capable source + counter + rebalance + comet-escape).
    -> angle,ships,commit (B,Ec).'''
    B, Ec = env.B, env.Ec
    dev = env.dev
    min_ships = 20.0
    angle = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    ships = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    commit = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    if opponent == 2:
        return angle, ships, commit
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == 1.0) & (env.p_alive > 0.5) & (half >= min_ships)
    if opponent == 0:  # random heading
        angle = torch.rand(B, Ec, dtype=DTYPE, device=dev) * (2.0 * PI)
        ships = torch.where(base, half, torch.zeros_like(half))
        commit = base.to(DTYPE)
        return angle, ships, commit
    if opponent == 3:  # medium (starter++): nearest non-owned planet INCL. rotating, lead-aimed at its intercept
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)             # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)            # (B,1,Ec) dest
        vt = (env.p_alive > 0.5) & (env.p_owner != 1.0)                 # (B,Ec) valid targets (rotating allowed)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
        dmask = torch.where(vt.unsqueeze(1), d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest valid target per source
        has_tgt = bestd < BIG * 0.5
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if LEAD_TARGET:                                                 # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(half.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        go = base & has_tgt
        ships = torch.where(go, half, torch.zeros_like(half))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 4:  # greedy: capture the NEAREST non-owned planet within CAPTURE_RADIUS we can already beat
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)            # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)           # (B,1,Ec) dest
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,Ec,Ec)
        vt = (env.p_alive > 0.5) & (env.p_owner != 1.0)                # (B,Ec) non-owned alive
        can_beat = env.p_ships.unsqueeze(2) > env.p_ships.unsqueeze(1) # (B,src,dst) src ships > dst ships
        cand = vt.unsqueeze(1) & can_beat & (d <= CAPTURE_RADIUS)      # capturable & in range
        dmask = torch.where(cand, d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest capturable target
        has_tgt = bestd < BIG * 0.5
        tgt_ships = env.p_ships.gather(1, tgt)                          # (B,Ec)
        n = torch.minimum(env.p_ships, torch.floor(tgt_ships) + CAPTURE_MARGIN)  # just enough to capture (+ buffer)
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if LEAD_TARGET:                                                 # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(n.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        own_ok = (env.p_owner == 1.0) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
        go = own_ok & has_tgt & (n >= 1.0)
        ships = torch.where(go, n, torch.zeros_like(n))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 5:  # intermediate: in-flight-aware capture from nearest capable source (cascade) + rebalance
        owned = (env.p_owner == 1.0) & (env.p_alive > 0.5) & (env.p_ships > 0.0)   # (B,Ec) sources that can act
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,src,dst)
        srcrange = torch.arange(Ec, device=dev).view(1, Ec, 1)
        # capture (in-flight aware): per target, needed = garrison + enemy_inbound - friendly_inbound + margin
        ftgt, _feta = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
        f_ok = (env.f_alive > 0.5) & (ftgt >= 0)
        tslot_f = ftgt.clamp(0, Ec - 1)
        in_fr = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # friendly ships already inbound per planet
        in_en = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # enemy ships inbound per planet
        in_fr.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner == 1.0) & f_ok).to(DTYPE))
        in_en.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner == 0.0) & f_ok).to(DTYPE))
        vt = (env.p_alive > 0.5) & (env.p_owner != 1.0)
        needed_t = torch.floor(env.p_ships) + torch.ceil(in_en) - torch.floor(in_fr) + CAPTURE_MARGIN   # (B,Ec)
        open_t = vt & (needed_t > 0.0)                                 # not already covered by friendly inbound
        can_supply = env.p_ships.unsqueeze(2) >= needed_t.unsqueeze(1) # (B,src,dst) source can cover the requirement
        capable = owned.unsqueeze(2) & open_t.unsqueeze(1) & can_supply
        d_cap = torch.where(capable, d, torch.full_like(d, BIG))
        nearest_cap = d_cap.argmin(1)                                  # (B,dst) nearest CAPABLE source (cascade)
        target_has = d_cap.min(1).values < BIG * 0.5
        assigned = (nearest_cap.unsqueeze(1) == srcrange) & target_has.unsqueeze(1)
        d_asg = torch.where(assigned, d, torch.full_like(d, BIG))
        cap_d, cap_tgt = d_asg.min(2)                                  # (B,Ec) source's nearest assigned target
        cap_has = cap_d < BIG * 0.5
        cap_n = torch.minimum(env.p_ships, needed_t.gather(1, cap_tgt))   # send exactly the requirement
        # rebalance: feed the poorest owned planet WITHIN REBALANCE_RADIUS (local) when surplus exceeds a percentage
        eyeE = (torch.eye(Ec, device=dev, dtype=DTYPE).unsqueeze(0) > 0.5)
        reb_cand = owned.unsqueeze(1) & (d <= REBALANCE_RADIUS) & (~eyeE)   # (B,src,dst) owned, in-radius, not self
        reb_dst_ships = torch.where(reb_cand, env.p_ships.unsqueeze(1), torch.full_like(d, BIG))
        poor_ships, poor_idx = reb_dst_ships.min(2)                    # (B,Ec) poorest owned-in-radius per source
        has_poor = poor_ships < BIG * 0.5
        gap = env.p_ships - poor_ships
        reb_ok = owned & (~cap_has) & has_poor & (gap > REBALANCE_PCT * env.p_ships.clamp_min(1.0))
        reb_n = torch.floor(gap * 0.5)
        reb_tgt = poor_idx
        # comet escape (top priority): owned comets evacuate ALL ships to the nearest owned non-comet planet
        safe_dst = (owned & (env.p_is_comet < 0.5)).unsqueeze(1) & (~eyeE)   # (B,src,dst) dest owned non-comet, not self
        d_evac = torch.where(safe_dst, d, torch.full_like(d, BIG))
        evac_d, evac_tgt = d_evac.min(2)
        evac_ok = owned & (env.p_is_comet > 0.5) & (evac_d < BIG * 0.5) & (env.p_ships >= 1.0)
        # priority: comet-escape > capture > rebalance
        use_cap = (~evac_ok) & cap_has
        use_reb = (~evac_ok) & (~cap_has) & reb_ok
        tgt = torch.where(evac_ok, evac_tgt, torch.where(use_cap, cap_tgt, reb_tgt))
        nn = torch.where(evac_ok, env.p_ships, torch.where(use_cap, cap_n, torch.where(use_reb, reb_n, torch.zeros_like(reb_n))))
        go = evac_ok | use_cap | use_reb
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)
        if LEAD_TARGET:
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(nn.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        ships = torch.where(go, nn, torch.zeros_like(nn))
        commit = go.to(DTYPE)
        return angle, ships, commit
    # opponent == 1: starter -- fire at nearest static non-owned planet
    distc = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
    is_static = (distc + env.p_radius) >= ROTATION_RADIUS_LIMIT
    valid_tgt = is_static & (env.p_alive > 0.5) & (env.p_owner != 1.0)
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(valid_tgt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, bestidx = dmask.min(2)
    has_tgt = bestd < BIG * 0.5
    btx = env.p_x.gather(1, bestidx); bty = env.p_y.gather(1, bestidx)
    angle = torch.atan2(bty - env.p_y, btx - env.p_x)
    go = base & has_tgt
    ships = torch.where(go, half, torch.zeros_like(half))
    commit = go.to(DTYPE)
    return angle, ships, commit

## 9. Env -- `step` (one tick; v5 target decode = atan2 toward the destination planet)
Action is (dest_slot, phi) per planet; heading aims straight at the destination
(`atan2(p_dest - p_src)`). One launch per planet. Self-play opponent decodes identically.

In [ ]:
class StepOut: pass


def _decode_target(env, action, legal):
    '''Dirichlet ALLOCATION decode. action = A (B,Ec,Ec): row s = allocation of planet s's ships over
    destinations (self-diagonal = hold). ships[s,d] = floor(A[s,d]*S_s) for d!=s, to ALIVE dests, >=
    MIN_LAUNCH_SHIPS; the rest stay home. Budget is structural (rows sum to 1). Each launch aims at the
    orbiting intercept. Returns angle/ships/can (B,Ec,Ec), valid (B,), invalid (B,), launches (B,).'''
    B, Ec, dev = env.B, env.Ec, env.dev
    A = action                                                  # (B,Ec,Ec) allocation
    S = env.p_ships                                             # (B,Ec)
    eye = torch.eye(Ec, dtype=DTYPE, device=dev).unsqueeze(0)   # (1,Ec,Ec)
    A_send = A * (1.0 - eye)                                    # zero the self-diagonal (hold)
    raw = A_send * S.unsqueeze(2)                               # (B,Ec,Ec) would-be ships src->dst
    dest_alive = (env.p_alive > 0.5).to(DTYPE).unsqueeze(1)     # (B,1,Ec)
    gate = legal.to(DTYPE).unsqueeze(2) * dest_alive            # (B,Ec,Ec) legal source & alive dest
    n = torch.floor(raw) * gate
    n = torch.where(n >= float(MIN_LAUNCH_SHIPS), n, torch.zeros_like(n))   # drop dribbles -> stay home
    can = (n >= 1.0).to(DTYPE)
    # intercept heading for every (src,dst): src = row planet, dst = col planet
    spx = env.p_x.unsqueeze(2); spy = env.p_y.unsqueeze(2)      # (B,Ec,1) source
    dpx = env.p_x.unsqueeze(1); dpy = env.p_y.unsqueeze(1)      # (B,1,Ec) dest
    off = env.p_radius.unsqueeze(2) + 0.1                       # (B,Ec,1) source surface
    if LEAD_TARGET:
        rdx = dpx - CENTER; rdy = dpy - CENTER
        r_d = torch.sqrt(rdx * rdx + rdy * rdy)                 # (B,1,Ec)
        phi0 = torch.atan2(rdy, rdx)
        drot = (env.p_rotates > 0.5).unsqueeze(1)               # (B,1,Ec)
        w = torch.where(drot, env.ang_vel.view(B, 1, 1), torch.zeros(B, 1, 1, device=dev, dtype=DTYPE))
        v = fleet_speed_t(n.clamp_min(1.0), env.vmax)           # (B,Ec,Ec)
        t = (torch.sqrt((dpx - spx) ** 2 + (dpy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        for _ in range(16):
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            t = (torch.sqrt((ix - spx) ** 2 + (iy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        phi = phi0 + w * t
        ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
        angle = torch.atan2(iy - spy, ix - spx)                 # (B,Ec,Ec)
    else:
        angle = torch.atan2(dpy - spy, dpx - spx).expand(B, Ec, Ec).contiguous()
    valid = can.sum((1, 2))                                     # launches all go to alive dests
    invalid = torch.zeros(B, dtype=DTYPE, device=dev)           # below-min launches just stay home (no penalty); budget is structural
    launches = can.sum((1, 2))
    return angle, n, can, valid, invalid, launches


def spawn_comets(env):
    '''Spawn COMET_PAIRS point-symmetric comet pairs into free planet slots: neutral objects that
    sweep the inner region at COMET_SPEED and expire past COMET_EXPIRE_RADIUS. Capturable; produce
    COMET_PRODUCTION once owned. Velocity aims past the center at a random perihelion (avoids the sun).'''
    B, dev = env.B, env.dev
    for _pair in range(COMET_PAIRS):
        theta = torch.rand(B, device=dev) * (2.0 * PI)
        d_peri = COMET_PERI_MIN + torch.rand(B, device=dev) * (COMET_PERI_MAX - COMET_PERI_MIN)
        sgn = torch.where(torch.rand(B, device=dev) < 0.5, torch.ones(B, device=dev), -torch.ones(B, device=dev))
        ships = torch.randint(1, 100, (B, 4), device=dev).min(1).values.to(DTYPE)   # min-of-4 -> skewed low
        for _c in range(2):
            th = theta + (0.0 if _c == 0 else PI)                       # point-symmetric pair
            r0 = float(COMET_SPAWN_RADIUS)
            px = CENTER + r0 * torch.cos(th); py = CENTER + r0 * torch.sin(th)
            toward = torch.atan2(CENTER - py, CENTER - px)
            alpha = torch.asin((d_peri / r0).clamp(max=0.99)) * (sgn if _c == 0 else -sgn)
            vdir = toward + alpha
            vx = COMET_SPEED * torch.cos(vdir); vy = COMET_SPEED * torch.sin(vdir)
            free = env.p_alive < 0.5
            slot = torch.argmax(free.to(torch.int8), 1, keepdim=True)    # (B,1) first free slot
            has = free.any(1, keepdim=True)
            def _set(field, val):
                v = val.unsqueeze(1) if val.dim() == 1 else val
                field.scatter_(1, slot, torch.where(has, v, field.gather(1, slot)))
            _set(env.p_alive, torch.ones(B, device=dev))
            _set(env.p_owner, torch.full((B,), -1.0, device=dev))
            _set(env.p_x, px); _set(env.p_y, py); _set(env.p_init_x, px); _set(env.p_init_y, py)
            _set(env.p_comet_vx, vx); _set(env.p_comet_vy, vy)
            _set(env.p_ships, ships)
            _set(env.p_prod, torch.full((B,), float(COMET_PRODUCTION), device=dev))
            _set(env.p_radius, torch.full((B,), float(COMET_RADIUS), device=dev))
            _set(env.p_is_comet, torch.ones(B, device=dev))
            _set(env.p_rotates, torch.zeros(B, device=dev))


def env_step(env, ego_action, opponent, act_threshold, opp_action=None):
    '''One tick. ego_action (B,Ec,2)=[dest,phi] (v5 target). opp_action (B,Ec,2) for self-play,
    else scripted opponent.'''
    B, Ec, Fc = env.B, env.Ec, env.Fc
    if COMETS_ENABLED and int(env.step_ct[0].item()) in COMET_SPAWN_STEPS:   # hidden-schedule comet spawn
        spawn_comets(env)
    vmax = env.vmax
    dev = env.dev
    ego, enemy = 0, 1

    prod_ego0 = (env.p_prod * (env.p_owner == float(ego)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    prod_enemy0 = (env.p_prod * (env.p_owner == float(enemy)).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)).sum(1)
    ego_owned0 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    # --- decode ego action (v5 target: one launch per planet) ---
    legal = (env.p_owner == float(ego)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
    e_ang, e_shp, e_can, valid, invalid, launches = _decode_target(env, ego_action, legal)

    # --- opponent launches ---
    slot_idx = torch.arange(Ec, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Ec)
    src_mat = torch.arange(Ec, dtype=torch.long, device=dev).view(1, Ec, 1).expand(B, Ec, Ec).reshape(B, Ec * Ec)
    flatM = lambda x: x.reshape(B, Ec * Ec)                                   # (B,Ec,Ec) -> (B,Ec*Ec)
    e_ang_f, e_shp_f, e_can_f = flatM(e_ang), flatM(e_shp), flatM(e_can)
    e_ded = e_shp.sum(2)                                                      # (B,Ec) total sent per source
    if opp_action is None:
        o_ang, o_shp, o_can = opponent_action(env, opponent)                 # scripted: (B,Ec) one launch/planet
        o_ded = o_shp
        o_slot = slot_idx; o_ang_f, o_shp_f, o_can_f = o_ang, o_shp, o_can; Lo = Ec
    else:  # self-play: decode player-1 allocation (B,Ec,Ec) exactly like the ego decode
        legal1 = (env.p_owner == float(enemy)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
        o_ang_t, o_shp_t, o_can_t, _, _, _ = _decode_target(env, opp_action, legal1)
        o_ded = o_shp_t.sum(2)
        o_slot = src_mat; o_ang_f, o_shp_f, o_can_f = flatM(o_ang_t), flatM(o_shp_t), flatM(o_can_t); Lo = Ec * Ec

    # --- deduct ships from origin planets ---
    env.p_ships = env.p_ships - e_ded - o_ded

    # --- place fleets (ego first, then opponent) ---
    Le = Ec * Ec; Ltot = Le + Lo
    owner = torch.cat([torch.full((B, Le), float(ego), dtype=DTYPE, device=dev),
                       torch.full((B, Lo), float(enemy), dtype=DTYPE, device=dev)], 1)
    from_slot = torch.cat([src_mat, o_slot], 1)
    angle = torch.cat([e_ang_f, o_ang_f], 1)
    ships = torch.cat([e_shp_f, o_shp_f], 1)
    commit = torch.cat([e_can_f, o_can_f], 1)
    seq = (env.step_ct * float(Ltot + 1)).unsqueeze(1) + \
          torch.arange(Ltot, dtype=DTYPE, device=dev).unsqueeze(0)
    launch_fleets(env, owner, from_slot, angle, ships, commit, seq)

    # --- production ---
    env.p_ships = env.p_ships + env.p_prod * (env.p_owner != -1.0).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)

    # --- planet new positions (orbit) ---
    stepf = env.step_ct
    dxc = env.p_init_x - CENTER; dyc = env.p_init_y - CENTER
    r = torch.sqrt(dxc * dxc + dyc * dyc)
    ia = torch.atan2(dyc, dxc)
    ca = ia + env.ang_vel.unsqueeze(1) * stepf.unsqueeze(1)
    rot = env.p_rotates > 0.5
    cmt = env.p_is_comet > 0.5
    nx = torch.where(rot, CENTER + r * torch.cos(ca), torch.where(cmt, env.p_x + env.p_comet_vx, env.p_x))
    ny = torch.where(rot, CENTER + r * torch.sin(ca), torch.where(cmt, env.p_y + env.p_comet_vy, env.p_y))
    old_px, old_py = env.p_x, env.p_y

    # --- fleet movement + swept collision against planet paths ---
    falive = env.f_alive > 0.5
    speed = fleet_speed_t(env.f_ships, vmax)
    fox, foy = env.f_x, env.f_y
    fnx = fox + torch.cos(env.f_angle) * speed
    fny = foy + torch.sin(env.f_angle) * speed
    Ax = fox.unsqueeze(2); Ay = foy.unsqueeze(2)
    Bx = fnx.unsqueeze(2); By = fny.unsqueeze(2)
    P0x = old_px.unsqueeze(1); P0y = old_py.unsqueeze(1)
    P1x = nx.unsqueeze(1); P1y = ny.unsqueeze(1)
    rad = env.p_radius.unsqueeze(1)
    palive = (env.p_alive.unsqueeze(1) > 0.5)
    d0x = Ax - P0x; d0y = Ay - P0y
    dvx = (Bx - Ax) - (P1x - P0x); dvy = (By - Ay) - (P1y - P0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - rad * rad
    disc = b * b - 4.0 * a * c
    sq = torch.sqrt(disc.clamp_min(0.0))
    t1 = (-b - sq) / (2.0 * a)
    t2 = (-b + sq) / (2.0 * a)
    hit_quad = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    hit_lin = (a < 1e-12) & (c <= 0.0)
    hit = torch.where(a < 1e-12, hit_lin, hit_quad)
    hit = hit & palive & falive.unsqueeze(2)        # (no comet check: comets omitted)
    slotf = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, 1, Ec)
    order = torch.where(hit, slotf, torch.full_like(slotf, float(Ec)))
    fh_v, tgt_slot = order.min(2)
    has_hit = fh_v < float(Ec)

    # OOB / sun removal (point-to-segment to sun center)
    oob = (fnx < 0.0) | (fnx > BOARD_SIZE) | (fny < 0.0) | (fny > BOARD_SIZE)
    vx, vy, wx, wy = fox, foy, fnx, fny
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    tt = ((CENTER - vx) * (wx - vx) + (CENTER - vy) * (wy - vy)) / l2.clamp_min(1e-12)
    tt = tt.clamp(0.0, 1.0)
    prx = vx + tt * (wx - vx); pry = vy + tt * (wy - vy)
    sundist = torch.sqrt((CENTER - prx) ** 2 + (CENTER - pry) ** 2)
    sun_hit = (l2 > 0.0) & (sundist < SUN_RADIUS)
    sun_pt = torch.sqrt((CENTER - vx) ** 2 + (CENTER - vy) ** 2) < SUN_RADIUS
    sun_hit = torch.where(l2 > 0.0, sun_hit, sun_pt)

    remove_fleet = falive & (has_hit | oob | sun_hit)
    contributes = falive & has_hit

    # --- combat: scatter arriving ships per owner, then elementwise resolve ---
    arr0 = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    arr1 = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    cf = contributes.to(DTYPE)
    s0 = env.f_ships * cf * (env.f_owner == float(ego)).to(DTYPE)
    s1 = env.f_ships * cf * (env.f_owner == float(enemy)).to(DTYPE)
    tslot = tgt_slot.clamp(0, Ec - 1)
    arr0.scatter_add_(1, tslot, s0)
    arr1.scatter_add_(1, tslot, s1)
    has0 = arr0 > 0.0; has1 = arr1 > 0.0
    top = torch.maximum(arr0, arr1); second = torch.minimum(arr0, arr1)
    both = has0 & has1
    surv_ships = torch.where(both, top - second, top)
    tie = both & (arr0 == arr1)
    surv_ships = torch.where(tie, torch.zeros_like(surv_ships), surv_ships)
    surv_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
    surv_owner = torch.where(arr0 > arr1, torch.zeros_like(surv_owner), surv_owner)
    surv_owner = torch.where(arr1 > arr0, torch.ones_like(surv_owner), surv_owner)
    any_arr = has0 | has1
    apply = any_arr & (surv_ships > 0.0) & (env.p_alive > 0.5)
    same = (env.p_owner == surv_owner)
    reinforce = apply & same
    attack = apply & (~same)
    env.p_ships = torch.where(reinforce, env.p_ships + surv_ships, env.p_ships)
    after = env.p_ships - surv_ships
    flips = attack & (after < 0.0)
    env.p_ships = torch.where(attack, torch.where(after < 0.0, -after, after), env.p_ships)
    env.p_owner = torch.where(flips, surv_owner, env.p_owner)

    # --- apply planet positions; clear removed fleets ---
    env.p_x = nx; env.p_y = ny
    if COMETS_ENABLED:                                   # comet expiry: swept past the inner region -> remove (ships lost)
        cdist = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
        gone = (env.p_is_comet > 0.5) & (env.p_alive > 0.5) & (cdist > COMET_EXPIRE_RADIUS)
        keepc = (~gone).to(DTYPE)
        env.p_alive = env.p_alive * keepc; env.p_is_comet = env.p_is_comet * keepc
        env.p_comet_vx = env.p_comet_vx * keepc; env.p_comet_vy = env.p_comet_vy * keepc
        env.p_ships = env.p_ships * keepc
        env.p_owner = torch.where(gone, torch.full_like(env.p_owner, -1.0), env.p_owner)
    keep = falive & (~remove_fleet)
    keepf = keep.to(DTYPE)
    env.f_alive = keepf
    env.f_owner = env.f_owner * keepf
    env.f_x = fnx * keepf; env.f_y = fny * keepf
    env.f_angle = env.f_angle * keepf
    env.f_ships = env.f_ships * keepf

    env.step_ct = env.step_ct + 1.0

    ego_owned1 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    out = StepOut()
    out.invalid = invalid
    out.valid = valid
    out.launches = launches
    out.launched_ships = e_shp.sum((1, 2))   # total ships launched this step (ship-weighted launch reward)
    out.owned_launch_ships = (e_shp * ego_owned0.unsqueeze(1).to(DTYPE)).sum((1, 2))   # ships launched to ALREADY-OWNED planets
    out.captured = (ego_owned1 & (~ego_owned0)).to(DTYPE).sum(1)
    out.lost = (ego_owned0 & (~ego_owned1)).to(DTYPE).sum(1)
    out.captured_prod = (env.p_prod * (ego_owned1 & (~ego_owned0)).to(DTYPE)).sum(1)   # production of newly captured planets
    out.lost_prod = (env.p_prod * (ego_owned0 & (~ego_owned1)).to(DTYPE)).sum(1)
    return out

## 10. Policy net -- v5 target heads (mirrors `model/policy_net.cpp`)
Same trunk; heads swapped to `dest_head` (B,E,E categorical logits), `phi_mu` (B,E,1),
`phi_logstd` (B,E,1). `dest_head` out dim == `PLANET_CAP` (the dest is an obs-slot index).

In [ ]:
class TxEncoderLayer(nn.Module):
    '''Pre-LN transformer encoder block: x = x + MHA(LN(x)); x = x + MLP(LN(x)).
    Multi-head; masks dead-planet KEYS (matches the legacy single-head attention).'''
    def __init__(self, h, n_heads, mlp_ratio):
        super().__init__()
        assert h % n_heads == 0, "HIDDEN (%d) not divisible by N_HEADS (%d)" % (h, n_heads)
        self.h, self.nh, self.hd = h, n_heads, h // n_heads
        self.ln1 = nn.LayerNorm(h)
        self.q = nn.Linear(h, h); self.k = nn.Linear(h, h); self.v = nn.Linear(h, h); self.o = nn.Linear(h, h)
        self.ln2 = nn.LayerNorm(h)
        self.mlp_in = nn.Linear(h, mlp_ratio * h); self.mlp_out = nn.Linear(mlp_ratio * h, h)
    def forward(self, x, entity_mask):
        B, E, _ = x.shape
        xn = self.ln1(x)
        q = self.q(xn).view(B, E, self.nh, self.hd).transpose(1, 2)        # (B,nh,E,hd)
        k = self.k(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        v = self.v(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.hd)  # (B,nh,E,E)
        dead = (entity_mask < 0.5).view(B, 1, 1, E)
        scores = scores.masked_fill(dead, -1e9)
        ctx = torch.matmul(torch.softmax(scores, -1), v)                   # (B,nh,E,hd)
        ctx = ctx.transpose(1, 2).reshape(B, E, self.h)
        x = x + self.o(ctx)
        x = x + self.mlp_out(torch.relu(self.mlp_in(self.ln2(x))))
        return x


class TransformerTrunk(nn.Module):
    '''proj -> n_stem pre-LN ResNet-MLP -> n_layers transformer blocks -> n_head pre-LN
    ResNet-MLP -> LN. Emits per-planet features (B,E,h); drop-in for the legacy trunk.'''
    def __init__(self, F, h, n_heads, n_layers, mlp_ratio, n_stem, n_head):
        super().__init__()
        self.proj = nn.Linear(F, h)
        self.stem_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_stem)])
        self.layers = nn.ModuleList([TxEncoderLayer(h, n_heads, mlp_ratio) for _ in range(n_layers)])
        self.head_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_head)])
        self.ln_out = nn.LayerNorm(h)
    def forward(self, entities, entity_mask):
        tok = self.proj(entities)
        for i in range(len(self.stem_a)):
            tok = tok + self.stem_b[i](torch.relu(self.stem_a[i](self.stem_ln[i](tok))))
        for layer in self.layers:
            tok = layer(tok, entity_mask)
        for i in range(len(self.head_a)):
            tok = tok + self.head_b[i](torch.relu(self.head_a[i](self.head_ln[i](tok))))
        return self.ln_out(tok)


class PolicyNet(nn.Module):
    '''v5 TARGET actor-critic (mirrors model/policy_net.cpp, target_actor branch).

    Trunk (UNCHANGED from the continuous actor): per-planet proj + masked cross-planet
    self-attention + GLU + n residual blocks + a separate board-globals embedding.
    Heads (CHANGED): dest_head -> (B,E,E) destination-categorical logits; phi_mu -> (B,E,1)
    phi mean; phi_logstd -> (B,E,1) state-dependent phi log-std (or a shared param).
    The dest categorical is over the E=PLANET_CAP obs slots, so dest_head out dim == max_entities.
    '''
    def __init__(self, F, G, hidden, d_g, max_entities, n_res_blocks, use_glu, std_state_dependent,
                 logstd_min, logstd_max, init_mu_scale, init_phi_bias, use_attention=True,
                 value_res_blocks=2, arch="trunk", n_heads=8, n_tx_layers=4, tx_mlp_ratio=4,
                 n_stem_res=1, n_head_res=1):
        super().__init__()
        self.F, self.G, self.h, self.d_g, self.E = F, G, hidden, d_g, max_entities
        self.use_glu = use_glu
        self.use_attention = use_attention
        self.arch = arch
        self.K = LAUNCHES_PER_PLANET
        self.std_state_dependent = std_state_dependent
        self.logstd_min = logstd_min
        self.logstd_max = logstd_max  # mutable: the anneal mutates this each iter
        h = hidden
        if arch == "transformer":
            self.trunk = TransformerTrunk(F, h, n_heads, n_tx_layers, tx_mlp_ratio, n_stem_res, n_head_res)
        else:
            self.proj = nn.Linear(F, h)
            if use_attention:                               # cross-planet self-attention (toggleable)
                self.attn_q = nn.Linear(h, h); self.attn_k = nn.Linear(h, h)
                self.attn_v = nn.Linear(h, h); self.attn_o = nn.Linear(h, h)
                self.ln_attn = nn.LayerNorm(h)
            self.ln_out = nn.LayerNorm(h)
            if use_glu:
                self.glu_gate = nn.Linear(h, h); self.glu_val = nn.Linear(h, h); self.glu_out = nn.Linear(h, h)
            self.res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.ln_res = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_res_blocks)])
        self.g_embed = nn.Linear(G, d_g)
        # --- v5 target heads ---
        self.dest_head = nn.Linear(h + d_g, max_entities)  # (B,E,E) per-source allocation logits (self=hold)
        if DISCRETE_PHI:
            self.phi_cat = nn.Linear(h + d_g, LAUNCHES_PER_PLANET * N_PHI)  # (B,E,K,N_PHI) per-slot phi logits
        else:
            self.phi_mu = nn.Linear(h + d_g, 1)             # (B,E,1) phi mean
            if std_state_dependent:
                self.phi_logstd = nn.Linear(h + d_g, 1)     # (B,E,1) state-dependent phi log-std
            else:
                self.phi_logstd_param = nn.Parameter(torch.zeros(1))
        # value head (PPO critic): input proj -> pre-LN RESIDUAL MLP (same block as the trunk) -> readout
        self.val_in = nn.Linear(h + d_g, h)
        self.val_res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(value_res_blocks)])
        self.val_out = nn.Linear(h, 1)
        # calm init: few planets commit at start; destination ~uniform
        with torch.no_grad():
            if DISCRETE_PHI:
                self.phi_cat.weight.mul_(init_mu_scale)     # near-zero feature-driven logits at start
                self.phi_cat.bias.fill_(0.0)
                self.phi_cat.bias.view(self.K, N_PHI)[:, 1:].fill_(init_phi_bias)  # per-slot: bias toward noop (bucket 0)
            else:
                self.phi_mu.weight.mul_(init_mu_scale)
                self.phi_mu.bias.fill_(init_phi_bias)

    def forward(self, entities, entity_mask, action_mask, globals_):
        if self.arch == "transformer":
            tok = self.trunk(entities, entity_mask)
        else:
            tok = self.proj(entities)                       # (B,E,d)
            # masked single-head self-attention over planets (pre-norm) -- the only cross-planet mixing
            if self.use_attention:
                xn = self.ln_attn(tok)
                Q = self.attn_q(xn); Kt = self.attn_k(xn); Vt = self.attn_v(xn)
                scores = torch.matmul(Q, Kt.transpose(1, 2)) / math.sqrt(self.h)   # (B,E,E)
                dead_key = (entity_mask < 0.5).unsqueeze(1)     # (B,1,E)
                scores = scores.masked_fill(dead_key, -1e9)
                attn = torch.matmul(torch.softmax(scores, -1), Vt)
                tok = tok + self.attn_o(attn)
            if self.use_glu:
                tok = tok + self.glu_out(self.glu_val(tok) * torch.sigmoid(self.glu_gate(tok)))
            for i in range(len(self.res_a)):
                tok = tok + self.res_b[i](torch.relu(self.res_a[i](self.ln_res[i](tok))))
            tok = self.ln_out(tok)
        B, E = tok.shape[0], tok.shape[1]
        gp = torch.relu(self.g_embed(globals_))         # (B,d_g)
        gpb = gp.unsqueeze(1).expand(B, E, self.d_g)
        hh = torch.cat([tok, gpb], -1)                  # (B,E,d+d_g)
        # --- v5 target heads ---
        dest_logits = self.dest_head(hh)               # (B,E,E) per-source allocation logits (self-diag = hold)
        # value: masked-mean pool of trunk tokens over live planets + g'
        em = entity_mask.unsqueeze(-1)
        pooled = (tok * em).sum(1) / em.sum(1).clamp_min(1.0)
        vh = self.val_in(torch.cat([pooled, gp], -1))                  # (B,d)
        for i in range(len(self.val_res_a)):                           # residual blocks (skip per block)
            vh = vh + self.val_res_b[i](torch.relu(self.val_res_a[i](self.val_ln[i](vh))))
        value = self.val_out(vh).squeeze(-1)                           # (B,)
        if DISCRETE_PHI:
            return dest_logits, value
        phi_mean = self.phi_mu(hh)                       # (B,E,1)
        if self.std_state_dependent:
            phi_logstd = self.phi_logstd(hh)            # (B,E,1)
        else:
            phi_logstd = self.phi_logstd_param.view(1, 1, 1).expand(B, E, 1)
        phi_logstd = phi_logstd.clamp(self.logstd_min, self.logstd_max)
        return dest_logits, phi_mean, phi_logstd, value


def build_policy():
    return PolicyNet(F_DIM, G_DIM, HIDDEN, D_G, PLANET_CAP, N_RES_BLOCKS, USE_GLU, STD_STATE_DEP,
                     LOGSTD_MIN, LOGSTD_MAX, INIT_MU_SCALE, INIT_PHI_BIAS,
                     use_attention=USE_ATTENTION,
                     value_res_blocks=VALUE_RES_BLOCKS,
                     arch=ARCH, n_heads=N_HEADS, n_tx_layers=N_TX_LAYERS, tx_mlp_ratio=TX_MLP_RATIO,
                     n_stem_res=N_STEM_RES, n_head_res=N_HEAD_RES).to(DEVICE)


def compute_reach(ent, em):
    """Per-source destination reachability (B,E,E): 1 unless a straight src->dest shot is absorbed
    by the sun. Recovered from encoded planet positions so it is identical in rollout and update.
    Diagonal forced to 1; the alive-mask is applied in the distribution."""
    px = ent[..., 0] * BOARD_SIZE
    py = ent[..., 1] * BOARD_SIZE
    B, E = px.shape
    sx = px.unsqueeze(2); sy = py.unsqueeze(2)         # source (B,E,1)
    dx = px.unsqueeze(1) - sx                           # src->dest (B,E,E)
    dy = py.unsqueeze(1) - sy
    dist = torch.sqrt(dx * dx + dy * dy).clamp_min(1e-6)
    ux = dx / dist; uy = dy / dist
    cx = CENTER - sx; cy = CENTER - sy                  # source -> sun centre
    t = cx * ux + cy * uy                               # closest-approach distance along the ray
    perp2 = (cx * cx + cy * cy) - t * t
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = t - torch.sqrt((sr - perp2).clamp_min(0.0))
    blocked = (t >= 0.0) & (perp2 <= sr) & (t_sun >= 0.0) & (t_sun < dist)
    reach = (~blocked).to(DTYPE)
    eye = torch.eye(E, device=ent.device, dtype=DTYPE).unsqueeze(0)
    return reach * (1.0 - eye) + eye                    # source -> itself is always reachable


def recover_ships(ent):
    """Integer ship count per planet, recovered from feature b6 = log1p(ships)/log(1000)."""
    return torch.round(torch.expm1((ent[..., 6] * SHIP_LOG_DENOM).clamp_min(0.0)))


def _make_dist(net, ent, em, am, gl):
    '''Build the actor distribution. owned = legal source planets (= action_mask),
    alive = valid destination slots (= entity_mask), reach = sun-reachability mask.'''
    reach = compute_reach(ent, em) if REACH_MASK else None
    out = net(ent, em, am, gl)
    if DISCRETE_PHI:
        dest_logits, value = out
        dist = DiscreteTargetDist(dest_logits, owned=am, alive=em, reach=reach, kappa=ALLOC_KAPPA)
    else:
        dest_logits, phi_mean, phi_logstd, value = out
        dist = TargetActorDist(dest_logits, phi_mean, phi_logstd, owned=am, alive=em, reach=reach)
    return dist, value


@torch.no_grad()
def act(net, ent, em, am, gl, greedy=False):
    dist, value = _make_dist(net, ent, em, am, gl)
    action = dist.greedy() if greedy else dist.sample()
    return action, dist.log_prob(action), value


def evaluate(net, ent, em, am, gl, action):
    dist, value = _make_dist(net, ent, em, am, gl)
    return dist.log_prob(action), dist.entropy(), value

## 11. Rollout + v5 reward assembly + GAE (mirrors `rollout.cpp::collect_gpu`)
v5 reward channels: capture +/-30, first-50 VALID-launch +1, production-milestone +20
per ego total-ship doubling; outcome (win/loss decayed past step 100) scattered onto
the last active step. GAE; reward+outcome scaled by 1/100. Heartbeat logs R[o c d p].

In [ ]:
def settle(env):
    '''{s0, s1, side0_alive, side1_alive}: ships (planets+fleets) per side + alive flags.'''
    pa = env.p_alive > 0.5; fa = env.f_alive > 0.5
    o0 = (env.p_owner == 0.0) & pa; o1 = (env.p_owner == 1.0) & pa
    g0 = (env.f_owner == 0.0) & fa; g1 = (env.f_owner == 1.0) & fa
    s0 = (env.p_ships * o0.to(DTYPE)).sum(1) + (env.f_ships * g0.to(DTYPE)).sum(1)
    s1 = (env.p_ships * o1.to(DTYPE)).sum(1) + (env.f_ships * g1.to(DTYPE)).sum(1)
    return s0, s1, (o0.any(1) | g0.any(1)), (o1.any(1) | g1.any(1))


def collect_ppo(env, net, world_pool, cursor, stage, opp_snapshot, rng, opp_override=None):
    '''Collect one PPO iteration on `B` envs. Returns a trajectory dict + stats + cursor.
    v5 reward (docs/set-ups/1.md): outcome (win/loss decayed past DECAY_START) + capture +/-30
    + first-50 VALID-launch +1 + production-milestone +20/doubling.'''
    Bn, Ec, T = B, env.Ec, env.T
    nap = 2                                  # v5 target action: (dest, phi)
    dev = env.dev

    # opponent: stages 1-3 follow the curriculum; stage 4 is chosen by the Elo league and
    # passed in as opp_override = {"kind": noop|random|starter|snapshot, "net": <frozen net|None>}.
    _KIND2OPP = {"noop": 2, "random": 0, "starter": 1, "medium": 3, "greedy": 4, "intermediate": 5, "snapshot": 1}
    if opp_override is not None:
        kind = opp_override["kind"]
        opp_snapshot = opp_override.get("net")
    elif stage == 1:
        kind = "noop"
    elif stage == 2:
        kind = "random"
    else:
        kind = "starter"
    selfplay = (kind == "snapshot")
    opp = _KIND2OPP[kind]

    # PPO: maximize world diversity (no group sharing)
    worlds = [world_pool[(cursor + i) % len(world_pool)] for i in range(Bn)]
    cursor += Bn
    env.reset(worlds)

    # host buffers (CPU)
    ent_buf = torch.zeros(T, Bn, Ec, F_DIM)
    em_buf = torch.zeros(T, Bn, Ec); am_buf = torch.zeros(T, Bn, Ec)
    gl_buf = torch.zeros(T, Bn, G_DIM); act_buf = torch.zeros(T, Bn, Ec, Ec)
    oldlp_buf = torch.zeros(T, Bn); valid_buf = torch.zeros(T, Bn)
    rew_buf = torch.zeros(T, Bn); val_buf = torch.zeros(T, Bn); done_buf = torch.zeros(T, Bn)

    active = torch.ones(Bn, device=dev)
    outcome = torch.zeros(Bn, device=dev)
    inv_sum = torch.zeros(Bn, device=dev); lnch_sum = torch.zeros(Bn, device=dev)
    valid_sum = torch.zeros(Bn, device=dev); step_sum = torch.zeros(Bn, device=dev)
    # per-game accumulators / per-channel totals (real units, for the heartbeat)
    launch_paid = torch.zeros(Bn, device=dev)              # CAP 3: cumulative launch reward credited this game
    milestone_prev = torch.zeros(Bn, device=dev)
    captureR = torch.zeros(Bn, device=dev); launchR = torch.zeros(Bn, device=dev)
    prodMR = torch.zeros(Bn, device=dev)

    last_t = 0
    for t in range(T):
        last_t = t
        ent, em, am, gl = env_encode(env, 0)
        a_t, logp, value = act(net, ent, em, am, gl, greedy=False)
        ent_buf[t].copy_(ent, non_blocking=True); em_buf[t].copy_(em, non_blocking=True)
        am_buf[t].copy_(am, non_blocking=True); gl_buf[t].copy_(gl, non_blocking=True)
        act_buf[t].copy_(a_t, non_blocking=True); oldlp_buf[t].copy_(logp, non_blocking=True)
        valid_buf[t].copy_(active, non_blocking=True); val_buf[t].copy_(value, non_blocking=True)

        opp_act = None
        if selfplay:
            with torch.no_grad():
                o1e, o1m, o1a, o1g = env_encode(env, 1)
                opp_act, _, _ = act(net if opp_snapshot is None else opp_snapshot,
                                    o1e, o1m, o1a, o1g, greedy=False)
        out = env_step(env, a_t, opp, ACT_THRESHOLD, opp_act)
        a = active

        # --- v5 event reward channels (docs/set-ups/1.md), a-masked ---
        cap_t = a * (CAPTURE_REWARD * ((out.captured + CAPTURE_PROD_SCALE * out.captured_prod)
                                       - CAPTURE_LOSS_FRAC * (out.lost + CAPTURE_PROD_SCALE * out.lost_prod)))  # base*(1+scale*prod) per planet

        # settle BEFORE the milestone term so it sees this tick's ego ship count.
        s0, s1, side0, side1 = settle(env)
        # production-milestone: +20 each NEW ego total-ship doubling (monotonic, no claw-back).
        # m(s) = floor(log2(s/base)) + 1 for s >= base, else 0.
        m_now = torch.where(s0 >= PROD_MILESTONE_BASE,
                            torch.floor(torch.log2(s0.clamp_min(PROD_MILESTONE_BASE) / PROD_MILESTONE_BASE)) + 1.0,
                            torch.zeros_like(s0))
        mr_t = a * (PROD_MILESTONE_REWARD * (m_now - milestone_prev).clamp_min(0.0))
        milestone_prev = torch.maximum(milestone_prev, m_now)

        # dispatch reward: ship-weighted, two factors (non-ego vs ego dest), valid only for the first
        # LAUNCH_WINDOW steps, overall per-step cap; unspent window is paid on an early WIN (outcome block).
        if t < LAUNCH_WINDOW:
            _enemy_ships = out.launched_ships - out.owned_launch_ships             # ships to NON-ego planets (uncapped)
            _self_ships  = torch.clamp(out.owned_launch_ships, max=SELF_LAUNCH_CAP)  # ego-dest ships, CAP 1
            _disp_raw    = LAUNCH_REWARD * _enemy_ships + SELF_LAUNCH_REWARD * _self_ships  # two factors
            launch_t = a * torch.clamp(_disp_raw, max=LAUNCH_STEP_CAP)             # CAP 2: overall per-step cap
        else:
            launch_t = torch.zeros_like(cap_t)                                     # launch window closed
        launch_room = (LAUNCH_GAME_CAP - launch_paid).clamp_min(0.0)            # CAP 3: whole-game launch budget left
        launch_t = torch.minimum(launch_t, launch_room)                        # cap TOTAL launch reward over the game
        launch_paid = launch_paid + launch_t
        captureR = captureR + cap_t; prodMR = prodMR + mr_t; launchR = launchR + launch_t

        r_t = cap_t + mr_t + launch_t                                      # v5 per-step reward (capture + milestone + launch)
        rew_buf[t].copy_(r_t / PPO_REWARD_SCALE, non_blocking=True)         # PPO value-target rescale

        inv_sum = inv_sum + a * out.invalid
        lnch_sum = lnch_sum + a * out.launches
        valid_sum = valid_sum + a * out.valid
        step_sum = step_sum + a

        step_now = env.step_ct
        dead = ~(side0 & side1)                       # true terminal: a side wiped out
        term = (step_now >= float(T - 2)) | dead
        done_buf[t].copy_(((active > 0.5) & dead).to(DTYPE), non_blocking=True)
        newly = (active > 0.5) & term
        outcome = torch.where(newly, torch.sign(s0 - s1), outcome)
        active = torch.where(term, torch.zeros_like(active), active)
        if (t & 15) == 15 and active.sum().item() == 0.0:
            break

    # settle any env still alive at the step cap
    s0, s1, side0, side1 = settle(env)
    outcome = torch.where(active > 0.5, torch.sign(s0 - s1), outcome)

    # bootstrap V(s_T) for envs alive at the cap (dead envs: notdone=0, unused)
    with torch.no_grad():
        fe, fm, fa_, fg = env_encode(env, 0)
        _, _, vT = act(net, fe, fm, fa_, fg, greedy=False)
        bootstrap = (vT * active).cpu()

    if dev.type == "cuda":
        torch.cuda.synchronize()
    Tu = last_t + 1
    keep = valid_buf[:Tu].reshape(-1).nonzero().squeeze(-1)

    outc = outcome.cpu()
    rew = rew_buf[:Tu]; val = val_buf[:Tu]; done = done_buf[:Tu]; alive = valid_buf[:Tu]
    # outcome: WIN/LOSS FLAT for the first DECAY_START steps, then decay symmetrically; draw = 0.
    length = alive.sum(0)                                       # (B,) episode length
    len_eff = (length - DECAY_START_STEP).clamp_min(0.0)        # decay only after DECAY_START
    win_val = torch.pow(torch.full_like(outc, WIN_DECAY), len_eff) * WIN_BONUS
    loss_val = torch.pow(torch.full_like(outc, LOSS_DECAY), len_eff) * LOSS_PENALTY
    Ot = torch.where(outc > 0.0, win_val,
                     torch.where(outc < 0.0, -loss_val, torch.zeros_like(outc)))
    r_outcome_log = Ot.mean().item()                           # outcome only (before rescale)
    # launch cashout: a WIN before the launch window closes pays the unspent window IN FULL
    # (at the per-step cap) -> end the game early instead of stalling to farm launch reward.
    disp_cashout = torch.clamp(LAUNCH_WINDOW - length, min=0.0) * LAUNCH_STEP_CAP * (outc > 0.0).to(outc.dtype)
    disp_cashout = torch.minimum(disp_cashout, (LAUNCH_GAME_CAP - launch_paid.cpu()).clamp_min(0.0))  # CAP 3: respect whole-game launch cap
    launchR = launchR + disp_cashout.to(launchR.device)        # launch-reward channel
    Ot = (Ot + disp_cashout) / PPO_REWARD_SCALE                # rescale outcome + cashout together
    last_idx = (length - 1.0).clamp_min(0.0).long().unsqueeze(0)   # (1,B)
    rew.scatter_add_(0, last_idx, Ot.unsqueeze(0))
    # GAE backward
    adv = torch.zeros(Tu, Bn)
    A = torch.zeros(Bn); nextval = bootstrap.clone()
    for t in range(Tu - 1, -1, -1):
        al = alive[t]; notdone = 1.0 - done[t]
        delta = rew[t] + GAMMA * nextval * notdone - val[t]
        A = delta + GAMMA * GAE_LAMBDA * notdone * A
        adv[t] = A * al
        nextval = val[t]
        A = A * al
    ret_full = (adv + val).reshape(-1)
    adv_full = adv.reshape(-1)
    mean_return_log = (rew.sum(0).mean().item()) * PPO_REWARD_SCALE   # real units

    def sel(x):
        return x.index_select(0, keep)
    tb = {}
    tb["entities"] = sel(ent_buf[:Tu].reshape(Tu * Bn, Ec, F_DIM))
    tb["entity_mask"] = sel(em_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["action_mask"] = sel(am_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["globals"] = sel(gl_buf[:Tu].reshape(Tu * Bn, G_DIM))
    tb["action"] = sel(act_buf[:Tu].reshape(Tu * Bn, Ec, Ec))
    tb["old_logp"] = sel(oldlp_buf[:Tu].reshape(Tu * Bn))
    adv_s = sel(adv_full)
    tb["advantage"] = (adv_s - adv_s.mean()) / (adv_s.std() + 1e-8)    # normalize over kept transitions
    tb["returns"] = sel(ret_full)
    tb["n"] = keep.shape[0]

    step_total = step_sum.sum().item()
    stats = {
        "mean_return": mean_return_log,
        "mean_len": step_total / Bn,
        "win_rate": (outcome > 0.0).to(DTYPE).mean().item(),
        "score": ((outcome > 0.0).to(DTYPE).mean() + 0.5 * (outcome == 0.0).to(DTYPE).mean()).item(),
        "inv_per_step": (inv_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "lnch_per_step": (lnch_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "valid_per_step": (valid_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "transitions": tb["n"],
        # per-channel means (real units, before reward_scale) -- heartbeat R[o c p ln]
        "r_outcome": r_outcome_log,
        "r_capture": captureR.mean().item(),
        "r_launch": launchR.mean().item(),
        "r_milestone": prodMR.mean().item(),
    }
    return tb, stats, cursor

## 12. PPO update (mirrors `grpo_trainer.cpp::update`)

Clipped surrogate + `VF_COEF*MSE(value, returns) - ENT_COEF*entropy`; grad clip; skip a
non-finite step. Entropy is **per-component** (`/ (E*3K)`) so `ent_coef` behaves like standard
PPO. Reports approx-KL, clipfrac, sigma.

In [ ]:
_HALF_LOG2PIE_C = 1.4189385332046727

def policy_surrogate(logp, oldlp, adv, clip):
    ratio = torch.exp((logp - oldlp).clamp(-LOGRATIO_CLAMP, LOGRATIO_CLAMP))   # guardrail: bound ratio so it can't overflow
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    return -torch.minimum(unclipped, clipped).mean()


def ppo_update(net, opt, tb):
    N = tb["n"]
    mb = MINIBATCHES
    mbsize = N // mb
    s = {"total": 0.0, "policy": 0.0, "vf": 0.0, "entropy": 0.0, "sigma": 0.0,
         "approx_kl": 0.0, "clipfrac": 0.0, "grad_norm": 0.0}
    if mbsize == 0:
        return s
    E = tb["entities"].shape[1]
    nap = 2                                          # v5 target action: (dest, phi)
    nsteps = 0
    _kl_stop = False
    for _ in range(UPDATE_EPOCHS):
        perm = torch.randperm(N)
        for b in range(mb):
            mi = perm[b * mbsize:(b + 1) * mbsize]
            ent = tb["entities"].index_select(0, mi).to(DEVICE)
            em = tb["entity_mask"].index_select(0, mi).to(DEVICE)
            am = tb["action_mask"].index_select(0, mi).to(DEVICE)
            gl = tb["globals"].index_select(0, mi).to(DEVICE)
            act_mb = tb["action"].index_select(0, mi).to(DEVICE)
            oldlp = tb["old_logp"].index_select(0, mi).to(DEVICE)
            adv = tb["advantage"].index_select(0, mi).to(DEVICE)
            ret = tb["returns"].index_select(0, mi).to(DEVICE)

            logp, entropy, value = evaluate(net, ent, em, am, gl, act_mb)
            pol = policy_surrogate(logp, oldlp, adv, CLIP)
            vloss = F.mse_loss(value, ret)
            ent_b = entropy.mean() / float(E * nap)         # per-component entropy (scale-invariant)
            loss = pol + VF_COEF * vloss - ENT_COEF * ent_b

            opt.zero_grad()
            loss.backward()
            gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD_NORM)
            if torch.isfinite(gnorm):
                opt.step()

            with torch.no_grad():
                logratio = (logp - oldlp).clamp(-LOGRATIO_CLAMP, LOGRATIO_CLAMP)
                ratio = torch.exp(logratio)
                mb_kl = ((ratio - 1.0) - logratio).mean().item()       # k3 KL estimator (>=0)
                s["total"] += loss.item(); s["policy"] += pol.item(); s["vf"] += vloss.item()
                s["entropy"] += ent_b.item()
                if DISCRETE_PHI:
                    s["sigma"] += ent_b.item()                  # discrete: report mean per-component entropy
                else:
                    mean_logstd = ent_b.item() - _HALF_LOG2PIE_C  # approx phi log-std
                    s["sigma"] += math.exp(mean_logstd)
                s["approx_kl"] += mb_kl
                s["clipfrac"] += (torch.abs(ratio - 1.0) > CLIP).to(DTYPE).mean().item()
                s["grad_norm"] += float(gnorm)
            nsteps += 1
            if mb_kl > KL_TARGET:                            # KL guardrail: stop over-updating this iteration
                _kl_stop = True
                break
        if _kl_stop:
            break
    if nsteps:
        for k in s:
            s[k] /= nsteps
    return s

## 13. Train loop (Elo-league driven, no hand-coded stages)

Iteration-driven curriculum, logstd-cap anneal (phase 1: force cap `LOGSTD_MAX -> LOGSTD_MAX_END`
over `SIGMA_DECAY_ITERS`; phase 2: release to `LOGSTD_MAX_POST`), stage-4 self-play snapshot
(refreshed every `SELFPLAY_REFRESH` iters). Per-iter heartbeat: stage, real return, launches/step,
approx-KL, clipfrac, sigma. Returns logged history for plotting.

In [ ]:
import copy

# ---------------- self-play Elo league (PFSP snapshot pool) ------------------
def _freeze_snapshot(net):
    """Move to CPU, eval mode, no grad, opponent sigma cap -> a fixed sparring partner."""
    net = net.cpu().eval()
    for p in net.parameters():
        p.requires_grad_(False)
    net.logstd_max = LOGSTD_MAX_POST
    return net


def load_snapshot(path):
    """Load a save_ckpt() .pt into a frozen PolicyNet on CPU (uses the ckpt's own width/depth)."""
    blob = torch.load(path, map_location="cpu")
    cfg = blob.get("config", {})
    net = PolicyNet(F_DIM, G_DIM, cfg.get("HIDDEN", HIDDEN), cfg.get("D_G", D_G), PLANET_CAP,
                    cfg.get("N_RES_BLOCKS", N_RES_BLOCKS), cfg.get("USE_GLU", USE_GLU),
                    cfg.get("STD_STATE_DEP", STD_STATE_DEP), LOGSTD_MIN, LOGSTD_MAX,
                    INIT_MU_SCALE, INIT_PHI_BIAS, use_attention=cfg.get("USE_ATTENTION", True),
                    value_res_blocks=cfg.get("VALUE_RES_BLOCKS", 2),
                    arch=cfg.get("ARCH", "trunk"), n_heads=cfg.get("N_HEADS", 8),
                    n_tx_layers=cfg.get("N_TX_LAYERS", 4), tx_mlp_ratio=cfg.get("TX_MLP_RATIO", 4),
                    n_stem_res=cfg.get("N_STEM_RES", 1), n_head_res=cfg.get("N_HEAD_RES", 1))
    net.load_state_dict(blob["model"])
    return _freeze_snapshot(net)


class League:
    """Pool of frozen opponents (scripted anchors + policy snapshots) with Elo ratings.
    Snapshots are PFSP-sampled; the two scripted bots are anchored to pin the Elo scale."""
    def __init__(self, rng):
        self.rng = rng
        self.learner_elo = ELO_INIT
        self.advanced = False                                # curriculum: greedy/medium locked until the advance gate
        self.members = [
            {"label": "random",  "net": None, "kind": "random",  "elo": ELO_RANDOM,   "anchor": True, "pinned": True, "n": 0},
            {"label": "starter", "net": None, "kind": "starter", "elo": ELO_STARTER,  "anchor": True, "pinned": True, "n": 0},
            {"label": "greedy",  "net": None, "kind": "greedy",  "elo": ELO_GREEDY,   "anchor": True, "pinned": True, "n": 0},
            {"label": "medium",  "net": None, "kind": "medium",  "elo": ELO_MEDIUM,   "anchor": True, "pinned": True, "n": 0},
            {"label": "intermediate","net": None,"kind": "intermediate","elo": ELO_INTERMEDIATE,"anchor": True, "pinned": True, "n": 0},
        ]

    def snapshots(self):
        return [m for m in self.members if m["kind"] == "snapshot"]

    def anchor(self, kind):
        return next(m for m in self.members if m["kind"] == kind)

    def add_checkpoint(self, path, label=None, elo=None):
        m = {"label": label or os.path.basename(path), "net": load_snapshot(path), "kind": "snapshot",
             "elo": ELO_INIT if elo is None else elo, "anchor": False, "pinned": True, "n": 0}
        self.members.append(m)
        return m

    def add_learner_snapshot(self, net, it):
        snap = _freeze_snapshot(copy.deepcopy(net))
        self.members.append({"label": "it%d" % it, "net": snap, "kind": "snapshot",
                             "elo": self.learner_elo, "anchor": False, "pinned": False, "n": 0})
        self._prune()

    def _prune(self):
        autos = [m for m in self.members if m["kind"] == "snapshot" and not m["pinned"]]
        for m in autos[:max(0, len(autos) - LEAGUE_MAX_SNAPSHOTS)]:   # drop the oldest (FIFO)
            self.members.remove(m)

    def _p_learner_beats(self, m):
        return 1.0 / (1.0 + 10.0 ** ((m["elo"] - self.learner_elo) / 400.0))

    def _pfsp_weight(self, p):
        if PFSP_MODE == "hard":
            return (1.0 - p) ** PFSP_POWER + PFSP_FLOOR   # focus on opponents you can't yet beat
        if PFSP_MODE == "even":
            return p * (1.0 - p) + PFSP_FLOOR             # focus on evenly-matched opponents
        return 1.0                                        # uniform

    def sample(self, net=None):
        """PFSP-sample ONE opponent, weighted MATCH_ELO_W*Elo + MATCH_FP_W*footprint-distance from the
        learner (scripted anchors count as maximally distinct; snapshot fingerprints cached). Pure-Elo
        when net is None. PFSP_FLOOR keeps every member at nonzero probability (anti-forgetting)."""
        members = self.members
        elo_w = [self._pfsp_weight(self._p_learner_beats(m)) for m in members]
        ws = elo_w
        if net is not None and MATCH_FP_W > 0.0:
            try:
                lfp = self._fingerprint(net)
                fd = []
                for m in members:
                    if m["net"] is None:
                        fd.append(1.0)                                  # scripted anchor: maximally distinct
                    else:
                        if m.get("fp") is None:
                            m["fp"] = self._fingerprint(m["net"])
                        fd.append(_js_div(lfp[0], m["fp"][0]) + _js_div(lfp[1], m["fp"][1]))
                mxe = max(elo_w) or 1.0; mxf = max(fd) or 1.0
                ws = [MATCH_ELO_W * (elo_w[i] / mxe) + MATCH_FP_W * (fd[i] / mxf) + PFSP_FLOOR
                      for i in range(len(members))]
            except Exception:
                ws = elo_w
        if SCRIPT_MATCH_BOOST != 1.0:                            # concentrate compute on UNMASTERED scripted anchors
            ws = list(ws)
            for i, m in enumerate(members):
                if m["net"] is None and m["kind"] != "noop":     # scripted anchor (random/starter/medium)
                    mastered = (m["n"] >= SCRIPT_BOOST_MIN_GAMES) and (m.get("wr") or 0.0) >= SCRIPT_BOOST_DROP_WR
                    if not mastered:
                        ws[i] *= SCRIPT_MATCH_BOOST
        ws = list(ws)
        if not self.advanced:                                    # curriculum: greedy/medium locked until the advance gate
            for i, m in enumerate(members):
                if m["kind"] in ("greedy", "medium"):
                    ws[i] = 0.0
        tot = sum(ws)
        if tot <= 0.0:
            return self.rng.choice(members)
        r = self.rng.random() * tot
        c = 0.0
        for m, w in zip(members, ws):
            c += w
            if r <= c:
                return m
        return members[-1]

    def update_elo(self, m, learner_score):
        """learner_score in [0,1] = win + 0.5*draw fraction over the rollout. Anchors stay fixed.
        Also tracks an EMA of the learner's score vs this member (m['wr']) for the scripted-boost gate."""
        exp = self._p_learner_beats(m)
        delta = ELO_K * (learner_score - exp)
        self.learner_elo += delta
        m["n"] += 1
        m["wr"] = learner_score if m.get("wr") is None else (1.0 - WR_EMA_BETA) * m["wr"] + WR_EMA_BETA * learner_score
        if not m["anchor"]:
            m["elo"] -= delta

    def leaderboard(self):
        rows = sorted(self.members, key=lambda m: -m["elo"])
        s = "  league | learner Elo %.0f | %d members\n" % (self.learner_elo, len(self.members))
        for m in rows:
            tag = "[anchor]" if m["anchor"] else ("[pinned]" if m["pinned"] else "")
            wr = m.get("wr"); wrs = "wr %.2f" % wr if wr is not None else "wr  -- "
            if m["net"] is None and m["kind"] != "noop":
                boosted = not ((m["n"] >= SCRIPT_BOOST_MIN_GAMES) and (wr or 0.0) >= SCRIPT_BOOST_DROP_WR)
                tag += " x%.1f" % SCRIPT_MATCH_BOOST if boosted else " (mastered)"
            s += "    %-12s elo %6.0f  n=%-4d %-8s %s\n" % (m["label"], m["elo"], m["n"], wrs, tag)
        return s




def anneal_logstd_max(net, it):
    Nd = SIGMA_DECAY_ITERS
    if Nd > 0:
        if it < Nd:
            f = it / Nd
            cur = LOGSTD_MAX + (LOGSTD_MAX_END - LOGSTD_MAX) * f
        else:
            cur = LOGSTD_MAX_POST
    else:
        cur = LOGSTD_MAX
    cur = max(cur, LOGSTD_MIN)
    net.logstd_max = cur
    return cur


def save_ckpt(net, path, meta=None):
    blob = {"model": net.state_dict(),
            "config": {"HIDDEN": HIDDEN, "N_RES_BLOCKS": N_RES_BLOCKS, "D_G": D_G,
                       "PLANET_CAP": PLANET_CAP, "F_DIM": F_DIM, "G_DIM": G_DIM,
                       "ARCH": ARCH, "N_TX_LAYERS": N_TX_LAYERS, "N_HEADS": N_HEADS,
                       "TX_MLP_RATIO": TX_MLP_RATIO, "N_STEM_RES": N_STEM_RES, "N_HEAD_RES": N_HEAD_RES,
                       "USE_GLU": USE_GLU, "USE_ATTENTION": USE_ATTENTION, "VALUE_RES_BLOCKS": VALUE_RES_BLOCKS,
                   "STD_STATE_DEP": STD_STATE_DEP,
                       "target_actor": True}}
    if meta:
        blob.update(meta)
    torch.save(blob, path)


def train(total_iters=TOTAL_ITERS, log_every=1, resume_from=None):
    net = build_policy()
    opt = torch.optim.Adam(net.parameters(), lr=LR, eps=ADAM_EPS)
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    rng = random.Random(SEED * 2654435761 + 12345)
    start_it = 0
    best_wr = -1.0

    league = League(rng)
    for ck in LEAGUE_INIT_CKPTS:
        try:
            m = league.add_checkpoint(ck)
            print("league seeded with %s  <- %s" % (m["label"], ck))
        except Exception as e:
            print("  !! skipped league ckpt %s -> %r" % (ck, e))

    if resume_from:
        start_it, best_wr = load_train_state(net, opt, league, resume_from)
        print("resumed from %s -> global iter %d, best_wr %.2f, %d league members"
              % (resume_from, start_it, best_wr, len(league.members)))

    hist = {"iter": [], "return": [], "win_rate": [],
            "lnch_per_step": [], "approx_kl": [], "clipfrac": [], "sigma": [], "grad_norm": [],
            "r_outcome": [], "r_capture": [], "r_milestone": [], "r_launch": [],
            "learner_elo": []}

    pool, pool_round = None, -1
    for step in range(total_iters):
        it = start_it + step
        rnd = (it // WORLD_RESAMPLE_EVERY) if WORLD_RESAMPLE_EVERY > 0 else 0
        if rnd != pool_round:
            pool = make_world_pool(N_WORLDS, base_seed=SEED + rnd * N_WORLDS)
            cursor, pool_round = 0, rnd
            print("  [worlds] pool round %d @ it%d (base_seed %d)" % (rnd, it, SEED + rnd * N_WORLDS))
        if ELO_RECAL_EVERY > 0 and step > 0 and it % ELO_RECAL_EVERY == 0:  # re-ground league ELO on the (fresh) worlds
            league.recalibrate_elo(net, pool)
        t0 = time.time()
        anneal_logstd_max(net, it)            # no-op for the discrete actor; sigma anneal for continuous

        # grow the pool: snapshot the learner (after a short anchors-only warmup) every SELFPLAY_REFRESH iters
        if (it >= SNAPSHOT_WARMUP) and (it % max(1, SELFPLAY_REFRESH) == 0):
            league.add_learner_snapshot(net, it + 1)

        # pick ONE opponent for this rollout: PFSP over the whole league (anchors + snapshots)
        opp_member = league.sample(net)
        if opp_member["net"] is not None:
            opp_member["net"].to(DEVICE)
        opp_override = {"kind": opp_member["kind"], "net": opp_member["net"]}

        tb, rs, cursor = collect_ppo(env, net, pool, cursor, 0, None, rng, opp_override=opp_override)
        us = ppo_update(net, opt, tb)

        # Elo: update the learner (and the non-anchored opponent) from this match\'s score
        league.update_elo(opp_member, rs["score"])
        if (not league.advanced) and league.learner_elo >= ADVANCE_TRIGGER_ELO:   # curriculum: confirm with an all-anchor recal (fresh seeds)
            league.recalibrate_elo(net, make_world_pool(ELO_RECAL_ENVS, base_seed=SEED + 104729 + 7919 * (it + 1)), all_anchors=True)
            if league.learner_elo >= ADVANCE_CONFIRM_ELO:
                league.advanced = True
                print("  [advance] learner hit %.0f ELO; all-anchor recal %.0f >= %d -> UNLOCK greedy/medium" % (ADVANCE_TRIGGER_ELO, league.learner_elo, ADVANCE_CONFIRM_ELO))
            else:
                print("  [advance] learner hit %.0f ELO but all-anchor recal %.0f < %d -> stay in base regime" % (ADVANCE_TRIGGER_ELO, league.learner_elo, ADVANCE_CONFIRM_ELO))
        if opp_member["net"] is not None:
            opp_member["net"].to("cpu")          # keep only the live policy resident on the GPU

        dt = time.time() - t0
        sps = rs["transitions"] / max(dt, 1e-9)

        hist["iter"].append(it + 1)
        hist["return"].append(rs["mean_return"]); hist["win_rate"].append(rs["win_rate"])
        hist["lnch_per_step"].append(rs["lnch_per_step"]); hist["approx_kl"].append(us["approx_kl"])
        hist["clipfrac"].append(us["clipfrac"]); hist["sigma"].append(us["sigma"]); hist["grad_norm"].append(us["grad_norm"])
        hist["r_outcome"].append(rs["r_outcome"]); hist["r_capture"].append(rs["r_capture"])
        hist["r_milestone"].append(rs["r_milestone"]); hist["r_launch"].append(rs["r_launch"])
        hist["learner_elo"].append(league.learner_elo)

        if (it + 1) % log_every == 0 or step == 0 or step == total_iters - 1:
            # heartbeat: R[o c p ln] = outcome, capture, prod-milestone, launch (real units)
            print("it%4d | ret %8.2f wr %.2f | R[o %7.1f c %6.1f p %5.1f ln %5.1f] | "
                  "lnch/st %.2f valid/st %.2f | kl %.3f cf %.2f gn %.1f sig %.3f | "
                  "loss %7.3f (pol %.3f vf %.3f) | elo %5.0f vs %-10s | sps %5.0f"
                  % (it + 1, rs["mean_return"], rs["win_rate"],
                     rs["r_outcome"], rs["r_capture"], rs["r_milestone"], rs["r_launch"],
                     rs["lnch_per_step"], rs["valid_per_step"], us["approx_kl"], us["clipfrac"], us["grad_norm"],
                     us["sigma"], us["total"], us["policy"], us["vf"],
                     league.learner_elo, opp_member["label"], sps))

        # checkpoint: periodic + best-by-win-rate
        if ((it + 1) % max(1, CKPT_EVERY) == 0) or (step == total_iters - 1):
            save_ckpt(net, CKPT_PATH, meta={"iter": it + 1, "win_rate": rs["win_rate"]})
            save_train_state(net, opt, league, it + 1, best_wr, TRAIN_STATE_PATH)
            print(league.leaderboard(), end="")
        if rs["win_rate"] > best_wr:
            best_wr = rs["win_rate"]
            save_ckpt(net, BEST_CKPT_PATH, meta={"iter": it + 1, "win_rate": rs["win_rate"]})

    save_ckpt(net, CKPT_PATH, meta={"iter": start_it + total_iters, "win_rate": hist["win_rate"][-1]})
    save_train_state(net, opt, league, start_it + total_iters, best_wr, TRAIN_STATE_PATH)
    print("saved final checkpoint ->", CKPT_PATH, "| best win-rate %.2f ->" % best_wr, BEST_CKPT_PATH)
    print(league.leaderboard(), end="")
    import json as _json
    _json.dump({k: [float(x) for x in v] for k, v in hist.items()},
               open(os.path.join(CKPT_DIR, "metrics.json"), "w"))
    print("saved metrics ->", os.path.join(CKPT_DIR, "metrics.json"))
    globals()["LEAGUE"] = league      # also expose as a module global (robust if the caller drops it)
    return net, hist, league

## 14. Run training

This is the heavy cell. With SMOKE=True it is a short sanity run; flip SMOKE off (section 2)
and re-run for the full spec. On a Colab/Kaggle T4 the full run takes hours -- reduce B,
HIDDEN, N_RES_BLOCKS, or TOTAL_ITERS to fit your budget. Checkpoints are written each cadence.

In [ ]:
# ---- resumable training state: model + Adam moments + global iter + self-play league --------
def _league_state_dict(self):
    members = []
    for m in self.members:
        members.append({"label": m["label"], "kind": m["kind"], "elo": m["elo"],
                        "anchor": m["anchor"], "pinned": m["pinned"], "n": m["n"], "wr": m.get("wr"),
                        "model": (m["net"].state_dict() if m["net"] is not None else None)})
    return {"learner_elo": self.learner_elo, "members": members}

def _league_load_state_dict(self, sd):
    self.learner_elo = sd["learner_elo"]
    self.members = []
    for e in sd["members"]:
        net = None
        if e["model"] is not None:
            net = build_policy(); net.load_state_dict(e["model"]); net = _freeze_snapshot(net)
        m = {"label": e["label"], "kind": e["kind"], "net": net, "elo": e["elo"],
             "anchor": e["anchor"], "pinned": e["pinned"], "n": e["n"]}
        if e.get("wr") is not None:
            m["wr"] = e["wr"]
        self.members.append(m)

League.state_dict = _league_state_dict
League.load_state_dict = _league_load_state_dict


def save_train_state(net, opt, league, global_iter, best_wr, path):
    """Full resumable state: weights + Adam moments + global iter + best win-rate + self-play league."""
    torch.save({"model": net.state_dict(), "opt": opt.state_dict(), "league": league.state_dict(),
                "global_iter": int(global_iter), "best_wr": float(best_wr)}, path)


def load_train_state(net, opt, league, path):
    """Restore net/opt/league IN PLACE. Returns (start_iter, best_wr). Resume on the same device.
    Accepts a full train-state OR a plain save_ckpt weight (warm-start: weights only, fresh Adam/league)."""
    blob = torch.load(path, map_location=DEVICE, weights_only=False)
    net.load_state_dict(blob["model"])
    if "opt" in blob:
        opt.load_state_dict(blob["opt"])
    else:
        print("  [resume] %s has no optimizer state -> WARM-START (fresh Adam moments)" % os.path.basename(path))
    start_it = int(blob.get("global_iter", blob.get("iter", 0)))
    if "league" in blob:
        league.load_state_dict(blob["league"])
    else:  # warm-start: calibrate learner Elo vs anchors so the first snapshot is not stamped Elo 0
        rnd = (start_it // WORLD_RESAMPLE_EVERY) if WORLD_RESAMPLE_EVERY > 0 else 0
        print("  [resume] no league state -> calibrating learner Elo vs anchors")
        league.recalibrate_elo(net, make_world_pool(ELO_RECAL_ENVS, base_seed=SEED + rnd * N_WORLDS))
    return start_it, float(blob.get("best_wr", -1.0))


# ---- small improvements: persist league snapshots locally + periodic ELO re-grounding ---------
import math

def _league_add_learner_snapshot(self, net, it):
    snap = _freeze_snapshot(copy.deepcopy(net))
    m = {"label": "it%d" % it, "net": snap, "kind": "snapshot",
         "elo": self.learner_elo, "anchor": False, "pinned": False, "n": 0}
    self.members.append(m)
    self._prune()
    try:  # persist the weights as soon as the snapshot is generated (survives FIFO pruning)
        d = os.path.join(CKPT_DIR, "league_agents"); os.makedirs(d, exist_ok=True)
        save_ckpt(snap, os.path.join(d, m["label"] + ".pt"), meta={"iter": it, "elo": float(m["elo"])})
    except Exception as e:
        print("  !! league snapshot save failed: %r" % e)

League.add_learner_snapshot = _league_add_learner_snapshot


def _eval_score(net, anchor_kind, worlds, n_envs):
    """No-grad mean learner-score (win + 0.5*draw) of `net` vs a scripted anchor on `worlds` (greedy)."""
    OPP = {"random": 0, "starter": 1, "noop": 2, "medium": 3, "greedy": 4, "intermediate": 5}
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)])
    active = torch.ones(n_envs, device=DEVICE); score = torch.zeros(n_envs, device=DEVICE)
    def _sc(s0, s1):
        return torch.where(s0 > s1, torch.ones_like(s0), torch.where(s0 < s1, torch.zeros_like(s0), torch.full_like(s0, 0.5)))
    with torch.no_grad():
        for _ in range(EPISODE_STEPS):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(net, ent, em, am, gl, greedy=True)
            env_step(env, a_t, OPP[anchor_kind], ACT_THRESHOLD, None)
            s0, s1, a0, a1 = settle(env)
            term = (env.step_ct >= float(env.T - 2)) | (~(a0 & a1))
            newly = (active > 0.5) & term
            score = torch.where(newly, _sc(s0, s1), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s0, s1, _, _ = settle(env)
        score = torch.where(active > 0.5, _sc(s0, s1), score)
    return float(score.mean().item())


def _league_recalibrate_elo(self, net, worlds, n_envs=None, all_anchors=False):
    """Re-ground EVERY rating from win-rate vs the FIXED anchors on `worlds`. Removes online-update drift."""
    n_envs = n_envs or ELO_RECAL_ENVS
    _ADV = ("greedy", "medium")
    anchors = [m for m in self.members if m["anchor"] and (all_anchors or self.advanced or m["kind"] not in _ADV)]
    def est(scores):  # scores: [(anchor_elo, learner_score), ...] -> averaged ELO estimate
        vals = []
        for ae, sc in scores:
            sc = min(max(sc, 0.02), 0.98)
            vals.append(ae + 400.0 * math.log10(sc / (1.0 - sc)))
        return sum(vals) / len(vals)
    self.learner_elo = est([(a["elo"], _eval_score(net, a["kind"], worlds, n_envs)) for a in anchors])
    for m in self.members:
        if m["anchor"] or m["net"] is None:
            continue
        m["net"].to(DEVICE)
        m["elo"] = est([(a["elo"], _eval_score(m["net"], a["kind"], worlds, n_envs)) for a in anchors])
        m["net"].to("cpu"); m["n"] = 0
    print("    [elo recal] learner=%.0f | %s" % (self.learner_elo,
          " ".join("%s=%.0f" % (m["label"], m["elo"]) for m in self.members if not m["anchor"])))

League.recalibrate_elo = _league_recalibrate_elo


# ---- league pruning: keep the most behaviorally DISTINCT snapshots (replaces FIFO) ----------
def _league_probe_obs(self):
    '''Fixed seeded synthetic observation batch used to fingerprint snapshot behavior (cached).'''
    if getattr(self, '_probe', None) is None:
        gen = torch.Generator().manual_seed(0xC0FFEE)
        P = 16
        ent = torch.randn(P, PLANET_CAP, F_DIM, generator=gen)
        em = (torch.rand(P, PLANET_CAP, generator=gen) > 0.3).to(DTYPE)        # ~70% planets alive
        am = em * (torch.rand(P, PLANET_CAP, generator=gen) > 0.5).to(DTYPE)   # some ego-owned
        gl = torch.randn(P, G_DIM, generator=gen)
        self._probe = (ent, em, am, gl)
    return self._probe

def _league_fingerprint(self, net):
    '''Behavioral fingerprint = mean (dest, phi) action distribution on the fixed probe batch.'''
    ent, em, am, gl = self._probe_obs()
    dev = next(net.parameters()).device
    with torch.no_grad():
        out = net(ent.to(dev), em.to(dev), am.to(dev), gl.to(dev))
        dest_logits, phi_logits = out[0], out[1]
        dest = torch.softmax(dest_logits, -1).reshape(-1, dest_logits.shape[-1]).mean(0)  # (E,)
        phi = torch.softmax(phi_logits, -1).reshape(-1, phi_logits.shape[-1]).mean(0)      # (N_PHI,)
    return dest.cpu(), phi.cpu()

def _js_div(p, q):
    m = 0.5 * (p + q); eps = 1e-9
    kl = lambda a, b: (a * (torch.log(a + eps) - torch.log(b + eps))).sum()
    return float(0.5 * kl(p, m) + 0.5 * kl(q, m))

def _league_prune(self):
    '''Keep LEAGUE_MAX_SNAPSHOTS most distinct auto-snapshots: repeatedly drop the one whose
    NEAREST neighbor (sum of dest+phi Jensen-Shannon divergence) is closest = most redundant.
    Anchors / pinned ckpts and the NEWEST snapshot are always kept. FIFO fallback on error.'''
    autos = [m for m in self.members if m['kind'] == 'snapshot' and not m['pinned']]
    if len(autos) <= LEAGUE_MAX_SNAPSHOTS:
        return
    try:
        fps = [self._fingerprint(m['net']) for m in autos]
    except Exception as e:
        print('  !! league diversity-prune -> FIFO fallback: %r' % e)
        for m in autos[:len(autos) - LEAGUE_MAX_SNAPSHOTS]:
            self.members.remove(m)
        return
    def d(i, j):
        return _js_div(fps[i][0], fps[j][0]) + _js_div(fps[i][1], fps[j][1])
    newest = len(autos) - 1                      # most recently appended -> always keep
    keep = list(range(len(autos)))
    while len(keep) > LEAGUE_MAX_SNAPSHOTS:
        drop, drop_nn = None, float('inf')
        for i in keep:
            if i == newest:
                continue
            nn = min(d(i, j) for j in keep if j != i)
            if nn < drop_nn:
                drop_nn, drop = nn, i
        if drop is None:
            break
        keep.remove(drop)
    kept = set(keep)
    for idx, m in enumerate(autos):
        if idx not in kept:
            self.members.remove(m)
    print('  league diversity-prune: kept %d/%d snapshots (max behavioral distinctness)'
          % (len(kept), len(autos)))

League._probe_obs = _league_probe_obs
League._fingerprint = _league_fingerprint
League._prune = _league_prune


In [ ]:
net, hist, league = train(total_iters=TOTAL_ITERS, log_every=1, resume_from=RESUME_FROM)

## 14b. Save every league agent as weights

Dumps each league pool member to `CKPT_DIR/league_agents/` as a `save_ckpt` blob (model + config + Elo/label/games). Scripted anchors have no weights, so they are skipped. Reloadable later via `load_snapshot` (e.g. to seed a future run through `LEAGUE_INIT_CKPTS`).

In [ ]:
# Save all league leaderboard agents as weights (run after training).
import os
LEAGUE_OUT = os.path.join(CKPT_DIR, "league_agents")
os.makedirs(LEAGUE_OUT, exist_ok=True)
saved = 0
_lg = globals().get("league") or globals().get("LEAGUE")
if _lg is None:
    raise NameError("No league in scope. Re-run training as `net, hist, league = train(...)` "
                    "with the UPDATED train cell. NOTE: the pool is in-memory only -- a finished "
                    "run whose train() did not return/stash it cannot be recovered.")
for m in sorted(_lg.members, key=lambda mm: -mm["elo"]):
    if m["net"] is None:                       # scripted anchors (random/starter) -> no weights
        continue
    safe = "".join(c if c.isalnum() else "_" for c in m["label"])
    path = os.path.join(LEAGUE_OUT, "%s_elo%04d.pt" % (safe, round(m["elo"])))
    save_ckpt(m["net"], path, meta={"elo": m["elo"], "label": m["label"], "games": m["n"]})
    saved += 1
    print("saved %-12s elo %6.0f  n=%-4d -> %s" % (m["label"], m["elo"], m["n"], path))
print("saved %d league agents -> %s" % (saved, LEAGUE_OUT))

## 15. Plot the logged curves (return / win-rate)

In [ ]:
# Training-health curves: each metric on its OWN panel + scale (raw = faint, EMA = bold).
def _ema(y, a=0.15):
    out, m = [], None
    for v in y:
        m = float(v) if m is None else (1 - a) * m + a * float(v)
        out.append(m)
    return out

it = hist["iter"]
def _panel(ax, key, title, pct=False, symlog=False):
    y = hist[key]
    ax.plot(it, y, color="tab:blue", alpha=0.25, lw=0.8)
    ax.plot(it, _ema(y), color="tab:blue", lw=1.8)
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.3, lw=0.5)
    if pct: ax.set_ylim(-0.02, 1.02)
    if symlog: ax.set_yscale("symlog", linthresh=1e-3)

fig, ax = plt.subplots(3, 4, figsize=(20, 10))
_panel(ax[0, 0], "return", "episode return (real units)")
_panel(ax[0, 1], "win_rate", "win rate (vs sampled opp)", pct=True)
_panel(ax[0, 2], "learner_elo", "learner Elo")
_panel(ax[1, 0], "sigma", "exploration (sigma / mean entropy)")
_panel(ax[1, 1], "clipfrac", "PPO clip fraction")
_panel(ax[1, 2], "approx_kl", "approx KL (symlog)", symlog=True)
_panel(ax[0, 3], "grad_norm", "grad norm (pre-clip, symlog)", symlog=True)
ax[1, 3].axis("off"); ax[2, 3].axis("off")
_panel(ax[2, 0], "lnch_per_step", "launches / step")
_panel(ax[2, 1], "r_outcome", "reward: outcome channel")
for k, c in (("r_capture", "tab:green"), ("r_launch", "tab:orange"), ("r_milestone", "tab:purple")):
    ax[2, 2].plot(it, hist[k], color=c, alpha=0.20, lw=0.8)
    ax[2, 2].plot(it, _ema(hist[k]), color=c, lw=1.6, label=k.replace("r_", ""))
ax[2, 2].set_title("reward: dense channels"); ax[2, 2].grid(alpha=0.3, lw=0.5); ax[2, 2].legend(fontsize=7)
for a in ax[-1]:
    a.set_xlabel("iter")
fig.suptitle("Orbit Wars v5 -- training health (faint = raw, bold = EMA)", fontsize=12)
plt.tight_layout(); plt.show()

## 16. Optional shape smoke test (default OFF)

A tiny `B=4` dry run of env + policy that checks tensor shapes and runs a couple of steps.
Guarded by `RUN_SMOKE=False` so it does not run heavy work on import / "Run all".

In [ ]:
RUN_SMOKE = False  # set True to run the shape sanity check

if RUN_SMOKE:
    senv = GpuEnv(PLANET_CAP, FLEET_CAP, 8, SHIP_SPEED, DEVICE)
    sworlds = make_world_pool(4, base_seed=123)
    senv.reset(sworlds)
    snet = build_policy()
    ent, em, am, gl = env_encode(senv, 0)
    assert ent.shape == (4, PLANET_CAP, F_DIM), ent.shape
    assert gl.shape == (4, G_DIM), gl.shape
    a_t, logp, value = act(snet, ent, em, am, gl)
    assert a_t.shape == (4, PLANET_CAP, PLANET_CAP), a_t.shape   # Dirichlet allocation matrix (src, dst)
    assert logp.shape == (4,) and value.shape == (4,)
    assert torch.isfinite(logp).all(), "log_prob not finite (mask leak?)"
    print("encode/act OK | ent", tuple(ent.shape), "act", tuple(a_t.shape),
          "logp", tuple(logp.shape), "V", tuple(value.shape))
    for opp in (2, 0, 1):   # noop / random / starter
        out = env_step(senv, a_t, opp, ACT_THRESHOLD, None)
        print("step opp=%d | launches=%.0f valid=%.0f captured=%.0f"
              % (opp, out.launches.sum(), out.valid.sum(), out.captured.sum()))
    # self-play step
    o1e, o1m, o1a, o1g = env_encode(senv, 1)
    opp_a, _, _ = act(snet, o1e, o1m, o1a, o1g)
    out = env_step(senv, a_t, 1, ACT_THRESHOLD, opp_a)
    print("self-play step OK | step_ct", senv.step_ct[0].item())
    print("SMOKE OK")
else:
    print("smoke test disabled (set RUN_SMOKE=True to enable)")